 # **01A. Creating daily activity profiles for different categories of individuals**

 This code creates synthetic profiles of activities performed by selected categories of individuals (e.g., household head, spouse, child) from several household types, starting from the ISTAT/EUROSTAT time-use diary dataset.

 The code also creates Markov-chain transition matrices and initial probabilities, which are then used to generate those profiles.
 The initial synthetic profiles cover a full year. It is also possible to extract one weekly profile for each season.

 WARNING: if these selected weeks are changed, the weeks considered for the Space Cooling profile and other downstream profiles must also be changed.

 ## 1.1 Import libraries and read input data

In [14]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.animation as animation
import matplotlib.dates as mdates
from pathlib import Path
from itertools import product
from functools import lru_cache
import os
import datetime
import gzip
import pickle

import glob



# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")


Start time : 2026-07-02 16:23:10.724405


 ## 1.2 Selectable sections

 This section defines which parts of the code should be executed or skipped (for example, if the desired output, such as transition matrices, has already been obtained). Modify this before running the code.

In [15]:
# Construction of aggregated datasets
RUN_AGGR_DATASETS = False        # wheter you want to aggregate original TUS data in "aggregated" classes of activities
RUN_DEBUG_INPUTS = False         # wheter you want to compare aggregated dataframes with "original" one generated by the authors

# 0)
RUN_MATRICES = False            # creation of transition matrices for several individual classes
RUN_PLOTS = True               # plot activity profiles and consumption

# 1)
RUN_YEARLY_PROFILES = False     # full-year activity and place profiles for individual classes
RUN_WEEKLY_PROFILES = False      # activity profiles for individual classes for selected weeks of the year

 ## 1.3 Dictionaries

This section defines the dictionaries used to specify the activities performed and the colors assigned to them in plots.

In [16]:
"""
    ---    DESCRIPTION  ---
    
    Below are the values for household type and individual position within the household used in Time-Use-Survey data

    TIPONUCLEO_LIST:

    tipnu2	household type

    0	0 Single members  
    1	1 Couple with children  
    2	2 Couple without children  
    3	3 Male single parent  
    4	4 Female single parent  


    POSNUCLEO_LIST:

    rpnuc2	position within the household

    0	Single member  
    1	Household head  
    2	Spouse  
    3	Child  

    Example:

    household_type_list = [1] --> Couple with children
    household_position_list = [3] --> Child

    WARNING: since the original used dataset is for the Italian case, in the following activities and location are referred to Italian classes. In case of EUROSTAT data, you need to align the categories. In # you can find the corresponding english name for activities and locations.

"""


# These values are intentionally kept in Italian because they are ISTAT source-data labels. Search for "ENGLISH definition" for plausible classification for EUROSTAT data
aggregated_activities = [
    'Dormire/Riposo', 'Igiene e vestirsi', 'Mangiare', 'Lavare stoviglie', 'Pulizia casa',
    'Cucinare', 'Spostamenti', 'Attività fuori casa', 'Lavare bucato', 'Guardare TV/PC',
    'Sport', 'Contatti sociali', 'Salute', 'Lavoro', 'Lettura', 'Tempo libero a casa',
    'Uso PC', 'Ascoltare musica', 'Manutenzione domestica', 'Stiratura', 'Studio', 'Telefono', 'Altro'
]

activity_english_dict = {
    'Dormire/Riposo': 'Sleeping/Resting',
    'Igiene e vestirsi': 'Personal care & dressing',
    'Mangiare': 'Eating',
    'Lavare stoviglie': 'Washing dishes',
    'Pulizia casa': 'House cleaning',
    'Cucinare': 'Cooking',
    'Spostamenti': 'Travelling/Commuting',
    'Attività fuori casa': 'Activities outside home',
    'Lavare bucato': 'Doing laundry',
    'Guardare TV/PC': 'Watching TV/PC',
    'Sport': 'Sports',
    'Contatti sociali': 'Social interactions',
    'Salute': 'Health-related activities',
    'Lavoro': 'Work',
    'Lettura': 'Reading',
    'Tempo libero a casa': 'Leisure at home',
    'Uso PC': 'Using PC',
    'Ascoltare musica': 'Listening to music',
    'Manutenzione domestica': 'Home maintenance',
    'Stiratura': 'Ironing',
    'Studio': 'Studying',
    'Telefono': 'Phone use',
    'Altro': 'Other'
}

# activity_color_map = {
#     'Dormire/Riposo': 'midnightblue', 'Igiene e vestirsi': 'darkblue', 'Mangiare': 'royalblue',
#     'Lavare stoviglie': 'deepskyblue', 'Pulizia casa': 'lightblue', 'Cucinare': 'limegreen',
#     'Spostamenti': 'yellowgreen', 'Attività fuori casa': 'gold', 'Lavare bucato': 'darkorange',
#     'Guardare TV/PC': 'orangered', 'Sport': 'firebrick', 'Contatti sociali': 'darkred',
#     'Salute': 'darkviolet', 'Lavoro': 'blueviolet', 'Lettura': 'mediumorchid', 'Tempo libero a casa': 'deeppink',
#     'Uso PC': 'hotpink', 'Ascoltare musica': 'pink', 'Manutenzione domestica': 'darkgreen', 'Stiratura': 'seagreen',
#     'Studio': 'lightseagreen', 'Telefono': 'teal', 'Altro': 'darkgrey'
# }

activity_color_map = {
    # Rest / personal routines
    'Dormire/Riposo': '#1b2a6b',
    'Igiene e vestirsi': '#4c72b0',
    'Mangiare': '#86b6f6',

    # Domestic work
    'Lavare stoviglie': '#56b4e9',
    'Pulizia casa': '#b3d7e8',
    'Cucinare': '#009e73',
    'Lavare bucato': '#e69f00',
    'Stiratura': '#44aa99',
    'Manutenzione domestica': '#117733',

    # Mobility / outside home
    'Spostamenti': '#7fbc41',
    'Attività fuori casa': '#f0e442',

    # Leisure / media
    'Guardare TV/PC': '#d55e00',
    'Tempo libero a casa': '#dd3497',
    'Uso PC': '#f781bf',
    'Ascoltare musica': '#fbb4ae',
    'Lettura': '#b07cc6',
    'Telefono': '#2a9d8f',

    # Social / health / sport
    'Sport': '#cc6677',
    'Contatti sociali': '#882255',
    'Salute': '#aa4499',

    # Work / study
    'Lavoro': '#6a3d9a',
    'Studio': '#88ccee',

    # Residual class
    'Altro': '#999999'
}



aggregated_places = [
    'Casa propria', 'Casa propria, spazi aperti', 'Seconda casa', 'Casa di altre persone', 'Albergo e simili',
    'Luogo di lavoro', 'Luogo formazione', 'Luogo chiuso: pasti', 'Luogo chiuso: commissioni', 'Luogo all\'aperto',
    'Luogo chiuso: tempo libero', 'Luogo di culto', 'Luogo n/d', 'Spostamento: net-zero', 'Spostamento: moto',
    'Spostamento: auto', 'Spostamento: HDV', 'Spostamento: acqua', 'Spostamento: TP', 'Spostamento: aria',
    'Altro luogo', 'Spostamento: n/d'
]


place_english_dict = {
    'Casa propria': 'Own home',
    'Casa propria, spazi aperti': 'Own home, outdoor spaces',
    'Seconda casa': 'Second home',
    'Casa di altre persone': 'Other people’s home',
    'Albergo e simili': 'Hotel and similar',
    'Luogo di lavoro': 'Workplace',
    'Luogo formazione': 'Place of education/training',
    'Luogo chiuso: pasti': 'Indoor place: meals',
    'Luogo chiuso: commissioni': 'Indoor place: errands',
    'Luogo all\'aperto': 'Outdoor place',
    'Luogo chiuso: tempo libero': 'Indoor place: leisure',
    'Luogo di culto': 'Place of worship',
    'Luogo n/d': 'Place n/a',
    'Spostamento: net-zero': 'Travel: walking/biking',
    'Spostamento: moto': 'Travel: motorcycle',
    'Spostamento: auto': 'Travel: car',
    'Spostamento: HDV': 'Travel: heavy-duty vehicle',
    'Spostamento: acqua': 'Travel: water transport',
    'Spostamento: TP': 'Travel: public transport',
    'Spostamento: aria': 'Travel: air transport',
    'Altro luogo': 'Other place',
    'Spostamento: n/d': 'Travel: n/a'
}

place_color_map = {
    # Home-related places
    'Casa propria': '#1b2a6b',
    'Casa propria, spazi aperti': '#4c72b0',
    'Seconda casa': '#86b6f6',
    'Casa di altre persone': '#8dd3c7',

    # Accommodation / work / education
    'Albergo e simili': '#b3b3b3',
    'Luogo di lavoro': '#6a3d9a',
    'Luogo formazione': '#88ccee',

    # Indoor public/commercial places
    'Luogo chiuso: pasti': '#e69f00',
    'Luogo chiuso: commissioni': '#f0e442',
    'Luogo chiuso: tempo libero': '#d55e00',
    'Luogo di culto': '#cc6677',

    # Outdoor / undefined
    'Luogo all\'aperto': '#009e73',
    'Luogo n/d': '#999999',
    'Altro luogo': '#666666',

    # Travel modes
    'Spostamento: net-zero': '#7fbc41',
    'Spostamento: moto': '#44aa99',
    'Spostamento: auto': '#117733',
    'Spostamento: HDV': '#332288',
    'Spostamento: acqua': '#56b4e9',
    'Spostamento: TP': '#0072b2',
    'Spostamento: aria': '#882255',
    'Spostamento: n/d': '#bbbbbb'
}

# place_color_map = {
#     'Casa propria': 'darkblue', 'Casa propria, spazi aperti': 'royalblue', 'Seconda casa': 'limegreen',
#     'Casa di altre persone': 'gold', 'Albergo e simili': 'darkorange', 'Luogo di lavoro': 'firebrick',
#     'Luogo formazione': 'darkred', 'Luogo chiuso: pasti': 'darkviolet', 'Luogo chiuso: commissioni': 'blueviolet',
#     'Luogo all\'aperto': 'deeppink', 'Luogo chiuso: tempo libero': 'hotpink', 'Luogo di culto': 'seagreen',
#     'Luogo n/d': 'teal', 'Spostamento: net-zero': 'midnightblue', 'Spostamento: moto': 'darkblue',
#     'Spostamento: auto': 'royalblue', 'Spostamento: HDV': 'limegreen', 'Spostamento: acqua': 'gold',
#     'Spostamento: TP': 'darkorange', 'Spostamento: aria': 'firebrick', 'Altro luogo': 'darkred',
#     'Spostamento: n/d': 'blueviolet'
# }


# (ENGLISH definition) 
# activity_color_map = {
#     'Sleeping/Resting': 'midnightblue', 'Personal care & dressing': 'darkblue', 'Eating': 'royalblue',
#     'Washing dishes': 'deepskyblue', 'House cleaning': 'lightblue', 'Cooking': 'limegreen',
#     'Travelling/Commuting': 'yellowgreen', 'Activities outside home': 'gold', 'Doing laundry': 'darkorange',
#     'Watching TV/PC': 'orangered', 'Sports': 'firebrick', 'Social interactions': 'darkred',
#     'Health-related activities': 'darkviolet', 'Work': 'blueviolet', 'Reading': 'mediumorchid', 'Leisure at home': 'deeppink',
#     'Using PC': 'hotpink', 'Listening to music': 'pink', 'Home maintenance': 'darkgreen', 'Ironing': 'seagreen',
#     'Studying': 'lightseagreen', 'Phone use': 'teal', 'Other': 'darkgrey'
# }

# aggregated_places = [
#     'Own home', 'Own home, outdoor spaces', 'Second home', 'Other people’s home', 'Hotel and similar',
#     'Workplace', 'Place of education/training', 'Indoor place: meals', 'Indoor place: errands', 'Outdoor place',
#     'Indoor place: leisure', 'Place of worship', 'Place n/a', 'Travel: walking/biking', 'Travel: motorcycle',
#     'Travel: car', 'Travel: heavy-duty vehicle', 'Travel: water transport', 'Travel: public transport', 'Travel: air transport',
#     'Other place', 'Travel: n/a'
# ]

# place_color_map = {
#     'Own home': 'darkblue', 'Own home, outdoor spaces': 'royalblue', 'Second home': 'limegreen',
#     'Other people’s home': 'gold', 'Hotel and similar': 'darkorange', 'Workplace': 'firebrick',
#     'Place of education/training': 'darkred', 'Indoor place: meals': 'darkviolet', 'Indoor place: errands': 'blueviolet',
#     'Outdoor place': 'deeppink', 'Indoor place: leisure': 'hotpink', 'Place of worship': 'seagreen',
#     'Place n/a': 'teal', 'Travel: walking/biking': 'midnightblue', 'Travel: motorcycle': 'darkblue',
#     'Travel: car': 'royalblue', 'Travel: heavy-duty vehicle': 'limegreen', 'Travel: water transport': 'gold',
#     'Travel: public transport': 'darkorange', 'Travel: air transport': 'firebrick', 'Other place': 'darkred',
#     'Travel: n/a': 'blueviolet'
# }


# Dictionaries for periods of the year (days, quarters, individual classes)
# These values are intentionally kept in Italian because they are ISTAT source-data labels.
source_day_types = [
    'feriale', 
    'sabato', 
    'domenica'
    ]

day_english_dict = {
    'feriale': 'Working day', 
    'sabato': 'Saturday', 
    'domenica': 'Sunday'
    }

# These values are intentionally kept in Italian because the quarter abbreviations are matched to the ISTAT source-data labels.
source_quarters = [
    'Novembre, Dicembre, Gennaio', 
    'Febbraio, Marzo, Aprile',
    'Maggio, Giugno, Luglio', 
    'Agosto, Settembre, Ottobre'
    ]

quarter_abbr_english_dict = {
    'Nov-Dic-Gen': 'Nov-Dec-Jan', 
    'Feb-Mar-Apr': 'Feb-Mar-Apr',
    'Mag-Giu-Lug': 'May-Jun-Jul', 
    'Ago-Set-Ott': 'Aug-Sep-Oct'   
    }

# Still to use Italian names if used ISTAT (i.e. Italian) data
quarter_abbreviation_map = {
    "Novembre, Dicembre, Gennaio": "Nov-Dic-Gen",
    "Febbraio, Marzo, Aprile": "Feb-Mar-Apr",
    "Maggio, Giugno, Luglio": "Mag-Giu-Lug",
    "Agosto, Settembre, Ottobre": "Ago-Set-Ott",
    "Nov-Dic-Gen": "Nov-Dic-Gen",
    "Feb-Mar-Apr": "Feb-Mar-Apr",
    "Mag-Giu-Lug": "Mag-Giu-Lug",
    "Ago-Set-Ott": "Ago-Set-Ott",
}
# quarter_abbreviation_map = {
#     "Novembre, Dicembre, Gennaio": "Nov-Dec-Jan",
#     "Febbraio, Marzo, Aprile": "Feb-Mar-Apr",
#     "Maggio, Giugno, Luglio": "May-Jun-Jul",
#     "Agosto, Settembre, Ottobre": "Aug-Sep-Oct",
#     "Nov-Dic-Gen": "Nov-Dec-Jan",
#     "Feb-Mar-Apr": "Feb-Mar-Apr",
#     "Mag-Giu-Lug": "May-Jun-Jul",
#     "Ago-Set-Ott": "Aug-Sep-Oct",
#     }


suffix_dict = {
    (0, 0): 'membro_isolato',
    (1, 1): 'coppiaCF_caponucleo' ,
    (1, 2): 'coppiaCF_coniuge',
    (1, 3): 'coppiaCF_figlio', 
    (2, 1): 'coppiaSF_caponu', 
    (2, 2): 'coppiaSF_coniug', 
    (3, 1): 'Mono_M_padre', 
    (3, 3): 'Mono_M_figlio',
    (4, 1): 'Mono_F_madre',
    (4, 3): 'Mono_F_figlio'    
    }

suffix_english_dict = {
    (0, 0): 'single',
    (1, 1): 'couple_w_child_head' ,
    (1, 2): 'couple_w_child_spouse',
    (1, 3): 'couple_w_child_child', 
    (2, 1): 'couple_wo_child_head', 
    (2, 2): 'couple_wo_child_spouse', 
    (3, 1): 'mono_M_father', 
    (3, 3): 'mono_M_child',
    (4, 1): 'mono_F_mother',
    (4, 3): 'mono_F_child'    
    }

# suffix_english_dict = {
#     'membro_isolato': 'single',
#     'coppiaCF_caponucleo': 'couple_w_child_head',
#     'coppiaCF_coniuge': 'couple_w_child_spouse',
#     'coppiaCF_figlio': 'couple_w_child_child', 
#     'coppiaSF_caponu': 'couple_wo_child_head', 
#     'coppiaSF_coniug': 'couple_wo_child_spouse', 
#     'Mono_M_padre': 'mono_M_father', 
#     'Mono_M_figlio': 'mono_M_child',
#     'Mono_F_madre': 'mono_F_mother',
#     'Mono_F_figlio': 'mono_F_child'
#     }

italy_regions_dict = {
    '1': 'Nord-ovest',
    '2': 'Nord-est',
    '3': 'Centro',
    '4': 'Sud',
    '5': 'Isole'    
}

region_abbr_dict = {
    'Nord-ovest': 'N-O',
    'Nord-est': 'N-E',
    'Centro': 'Cen',
    'Sud': 'Sud',
    'Isole': 'Isl'    
}


quarter_abbr_dict = {
    'Novembre, Dicembre, Gennaio': 'Nov-Dic-Gen',
    'Febbraio, Marzo, Aprile': 'Feb-Mar-Apr',
    'Maggio, Giugno, Luglio': 'Mag-Giu-Lug',
    'Agosto, Settembre, Ottobre': 'Ago-Set-Ott'
}

# Define region groups using the keys of italy_regions_dict
region_groups = {
    'Group1': ['1', '2', '3'],  # 'Nord-ovest', 'Nord-est', 'Centro'
    'Group2': ['4', '5']        # 'Sud', 'Isole'
}

# Define the aggregated macro-regions here
region_group_names = {'N-O_N-E_Cen', 'Sud_Isl'}


# Map the English day_name to the Italian source/file label
source_day_map = {
    'Saturday':  'sabato',
    'Sunday':    'domenica'
    # all other days -> 'feriale' (legacy ISTAT/source label)
}

 ## 1.4 Simulation parameters

 Defined the most important parameters (e.g., which individual classes to characterize, number of individuals for whom to recreate a synthetic profile, profile duration, and weeks to extract from the annual profile).

In [17]:
# Household-type + household-position configurations to analyze:
# individual_configurations = [(0, 0), (1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (3, 1), (3, 3), (4, 1), (4, 3)]      # e.g., (1,3) --> household type = couple with children, member = child
individual_configurations = [(0,0)]

# Number of people for whom to create a synthetic profile
simulated_people = 200

# Time parameters (WARNING: if the selected periods are changed, the COOLING profiles and other downstream profiles must also be updated!) 
simulation_year = 2023     
simulation_days = 365
num_time_steps = 144        # n° of timestep daily (10' time resolution)

# Weeks for which profiles are extracted
selected_periods = {
    'Winter profiles':      ('2023-02-15 00:00:00', '2023-02-21 23:50:00'),
    'Spring profiles':      ('2023-04-19 00:00:00', '2023-04-25 23:50:00'),  
    'Summer profiles':      ('2023-07-12 00:00:00', '2023-07-18 23:50:00'), 
    'Autumn profiles':      ('2023-10-11 00:00:00', '2023-10-17 23:50:00'),
    'Final winter day':     ('2023-12-20 00:00:00', '2023-12-20 23:50:00'),
}

## 1.5 Directories
Place where all directories for folders and generated files

In [ ]:
# ------------------
#   WARNING: Need to adjust the directories to the folder of the present code
# ------------------
# Define the output folder and subfolders
# --- Inputs
input_folder = os.path.join('input\\01_input_profiles_att')
istat_original_tus_data_folder = os.path.join(input_folder, 'ISTAT original TUS data')

# --- Outputs
output_folder = 'output'
output_pictures_folder = os.path.join(output_folder, '01_profiles_Pics')
        
activity_matrices_folder = Path("output") / "01_Mtx_activ"
location_matrices_folder = Path("output") / "01_Mtx_places"
yearly_activity_profiles_folder = Path("output") / "01_activity_profiles" / "yearly_profiles"
weekly_activity_profiles_folder = Path("output") / "01_activity_profiles" / "weekly_profiles"

# Directory for old/original input files provided by the authors
rebuilt_input_folder = Path(input_folder)




# Create folders if they do not exist
for folder in [
    output_pictures_folder,
    input_folder,
    istat_original_tus_data_folder,
    activity_matrices_folder,
    location_matrices_folder,
    yearly_activity_profiles_folder,
    weekly_activity_profiles_folder,
]:
    os.makedirs(folder, exist_ok=True)     # , output_python_based_folder, output_excel_based_folder]:

 ## 1.6 Function definitions

 Defined all required functions here (for example, to apply filters to the dataframe considered, calculate transition matrices, etc.).

## 1.6.1 F(x) for rebuilding aggregated datasets

In [19]:
# ----------------------------------------------------------
#       Sec 1.6 -- 0) FUNCTIONS FOR REBUILDING AGGREGATED DATASETS
# ----------------------------------------------------------

# Function to save created picle compressed files
def save_pickle_gzip(obj, file_path):
    """Save a Python object as a gzip-compressed pickle file."""
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with gzip.open(file_path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


# Function to load pickle files
def load_pickle_gzip(file_path):
    """Load a Python object from a gzip-compressed pickle file."""
    with gzip.open(file_path, "rb") as f:
        return pickle.load(f)


# Function to align original TUS data with activity aggregation defined by the authors
def normalize_mapping_code(series, zfill=None, strip_leading_zeros=False):
    """
    Normalize ISTAT code columns to make source-data codes and legend-table codes comparable.
    """
    normalized_series = (
        series.astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    if strip_leading_zeros:
        normalized_series = normalized_series.str.lstrip("0")

    if zfill is not None:
        normalized_series = normalized_series.str.zfill(zfill)

    return normalized_series



# Function to adjust nomenclature for quarters in the file
def convert_quarter_labels_to_abbreviations(df):
    """
    Convert the original Italian quarter labels into the abbreviated labels used in this script.

    The rest of ENG_01 uses:
        - Nov-Dec-Jan
        - Feb-Mar-Apr
        - May-Jun-Jul
        - Aug-Sep-Oct
    """

    if "Trimestre" in df.columns:
        df["Trimestre"] = (
            df["Trimestre"]
            .astype(str)
            .str.strip()
            .map(quarter_abbreviation_map)
            .fillna(df["Trimestre"].astype(str).str.strip())
        )

    return df


# Function to rebuild "df_individui_full" dataframe with all information for individuals
def build_individuals_dataframe(input_dir):
    """
    Build the full individuals dataframe from the original ISTAT individuals file.

    Output:
        df_individui_full.pkl.gz
    """
    input_dir = Path(input_dir)

    individuals_file_path = Path(istat_original_tus_data_folder) / "uso_tempo_Microdati_Anno_2013_Individui.txt"       # original ISTAT TUS dataset (to be adjusted if used EUROSTAT one)
    legend_file_path = input_dir / "descrizione_dati_TUS.xlsx"                              # aggregation list for activities, places, individuals defined by the authors

    individuals_df = pd.read_csv(individuals_file_path, delimiter="\t", dtype=str)

    individuals_legend_df = pd.read_excel(legend_file_path, sheet_name="Legenda")

    # category definition (see the original ISTAT TUS data documentation for the meaning of each class)
    code_columns = [
        "rk3", "tipnu2", "rpnuc2", "tipfa2m", "tipoab",
        "distsc", "abitlo", "auto", "bic", "motor", "amoto"
    ]

    for column in code_columns:
        if column in individuals_df.columns:
            individuals_df[column] = individuals_df[column].astype(str).str.strip()

        if column in individuals_legend_df.columns:
            individuals_legend_df[column] = normalize_mapping_code(
                individuals_legend_df[column],
                strip_leading_zeros=True
            )

    # Description/mapping of each TUS column 
    diary_mapping = dict(zip(
        individuals_legend_df["rk3"],
        individuals_legend_df["Diario giornaliero"]
    ))

    household_type_mapping = dict(zip(
        individuals_legend_df["tipnu2"],
        individuals_legend_df["tipo di nucleo"]
    ))

    household_position_mapping = dict(zip(
        individuals_legend_df["rpnuc2"],
        individuals_legend_df["posizione nel nucleo familiare"]
    ))

    dwelling_type_mapping = dict(zip(
        individuals_legend_df["tipoab"],
        individuals_legend_df["tipologia abitazione"]
    ))

    study_distance_mapping = dict(zip(
        individuals_legend_df["distsc"],
        individuals_legend_df["Lontananza della sua abitazione dal luogo dove si svolge il corso"]
    ))

    work_distance_mapping = dict(zip(
        individuals_legend_df["abitlo"],
        individuals_legend_df["Distanza abitazione da luogo di lavoro"]
    ))

    car_ownership_mapping = dict(zip(
        individuals_legend_df["auto"],
        individuals_legend_df["la famiglia possiede: automobile"]
    ))

    bike_ownership_mapping = dict(zip(
        individuals_legend_df["bic"],
        individuals_legend_df["la famiglia possiede: bicicletta"]
    ))

    scooter_ownership_mapping = dict(zip(
        individuals_legend_df["motor"],
        individuals_legend_df["la famiglia possiede: motorino, scooter"]
    ))

    motorcycle_ownership_mapping = dict(zip(
        individuals_legend_df["amoto"],
        individuals_legend_df["la famiglia possiede: motocicletta, moto"]
    ))

    individuals_df["Diario giornaliero"] = individuals_df["rk3"].map(diary_mapping)
    individuals_df["tipo di nucleo"] = individuals_df["tipnu2"].map(household_type_mapping)
    individuals_df["posizione nel nucleo familiare"] = individuals_df["rpnuc2"].map(household_position_mapping)
    individuals_df["tipologia abitazione"] = individuals_df["tipoab"].map(dwelling_type_mapping)
    individuals_df["Lontananza della sua abitazione dal luogo dove si svolge il corso"] = individuals_df["distsc"].map(study_distance_mapping)
    individuals_df["Distanza abitazione da luogo di lavoro"] = individuals_df["abitlo"].map(work_distance_mapping)
    individuals_df["la famiglia possiede: automobile"] = individuals_df["auto"].map(car_ownership_mapping)
    individuals_df["la famiglia possiede: bicicletta"] = individuals_df["bic"].map(bike_ownership_mapping)
    individuals_df["la famiglia possiede: motorino, scooter"] = individuals_df["motor"].map(scooter_ownership_mapping)
    individuals_df["la famiglia possiede: motocicletta, moto"] = individuals_df["amoto"].map(motorcycle_ownership_mapping)

    # Unique person identifier used throughout the following dataframes.
    individuals_df["persona"] = (
        individuals_df["profam"].astype(str) + "." + individuals_df["proind"].astype(str)
    )

    # Move the person identifier to the first column.
    columns = individuals_df.columns.tolist()
    columns.insert(0, columns.pop(columns.index("persona")))
    individuals_df = individuals_df[columns]

    save_pickle_gzip(
        individuals_df,
        input_dir / "df_individui_full.pkl.gz"
    )

    return individuals_df



# Function to properly split activities that cross midnight (e.g., sleeping from 23:30 to 06:30)
def split_rows_crossing_midnight(df, start_column="ora inizio", end_column="ora fine"):
    """
    Split rows whose start time is later than their end time, meaning that
    the recorded activity/place crosses midnight.

    Example:
        23:30 -> 06:30

    becomes:
        23:30 -> 23:59
        00:00 -> 06:30
    """
    df = df.copy()

    df[start_column] = pd.to_datetime(
        df[start_column],
        format="%H:%M",
        errors="coerce"
    )

    df[end_column] = pd.to_datetime(
        df[end_column],
        format="%H:%M",
        errors="coerce"
    )

    crosses_midnight_mask = df[start_column] > df[end_column]

    if not crosses_midnight_mask.any():
        return df

    before_midnight_df = df.loc[crosses_midnight_mask].copy()
    after_midnight_df = df.loc[crosses_midnight_mask].copy()

    before_midnight_df[end_column] = pd.to_datetime("23:59", format="%H:%M")
    after_midnight_df[start_column] = pd.to_datetime("00:00", format="%H:%M")

    df = pd.concat(
        [
            df.loc[~crosses_midnight_mask],
            before_midnight_df,
            after_midnight_df
        ],
        ignore_index=True
    )

    return df



# Function to rebuild the full daily activity dataframe from ISTAT data
def build_daily_diary_dataframe(input_dir):
    """
    Build the full daily diary dataframe from the original ISTAT daily diary file.

    Output:
        df_diario_giornaliero_full.pkl.gz
    """
    input_dir = Path(input_dir)

    diary_file_path  = Path(istat_original_tus_data_folder) / "uso_tempo_Microdati_Anno_2013_DiarioGiornaliero.txt"     # original ISTAT TUS dataset (to be adjusted if used EUROSTAT one)
    legend_file_path = input_dir / "descrizione_dati_TUS.xlsx"                              # aggregation list for activities, places, individuals defined by the authors

    diary_df = pd.read_csv(
        diary_file_path,
        delimiter="\t",
        dtype=str
    )

    diary_legend_df = pd.read_excel(
        legend_file_path,
        sheet_name="Legenda"
    )

    # Normalize source-data codes.
    diary_df["cluogo"] = normalize_mapping_code(diary_df["cluogo"], zfill=3)
    diary_df["catpri"] = normalize_mapping_code(diary_df["catpri"], zfill=3)
    diary_df["meseri"] = diary_df["meseri"].astype(str).str.strip()
    diary_df["catcon"] = diary_df["catcon"].astype(str).str.strip()
    diary_df["gsett"] = diary_df["gsett"].astype(str).str.strip()

    # Normalize legend-table codes.
    diary_legend_df["cluogo"] = normalize_mapping_code(diary_legend_df["cluogo"], zfill=3)
    diary_legend_df["catpri"] = normalize_mapping_code(diary_legend_df["catpri"], zfill=3)
    diary_legend_df["meseri"] = normalize_mapping_code(diary_legend_df["meseri"])
    diary_legend_df["catcon"] = normalize_mapping_code(diary_legend_df["catcon"])
    diary_legend_df["gsett"] = normalize_mapping_code(diary_legend_df["gsett"])

    place_mapping = dict(zip(
        diary_legend_df["cluogo"],
        diary_legend_df["Luogo aggregato"]
    ))

    activity_mapping = dict(zip(
        diary_legend_df["catpri"],
        diary_legend_df["Attività aggregata"]
    ))

    quarter_mapping = dict(zip(
        diary_legend_df["meseri"],
        diary_legend_df["Trimestre"]
    ))

    condition_mapping = dict(zip(
        diary_legend_df["catcon"],
        diary_legend_df["Descrizione catcon"]
    ))

    day_type_mapping = dict(zip(
        diary_legend_df["gsett"],
        diary_legend_df["Descrizione gsett"]
    ))

    diary_df["Luogo aggregato"] = diary_df["cluogo"].map(place_mapping)
    diary_df["Attività aggregata"] = diary_df["catpri"].map(activity_mapping)
    diary_df["Trimestre"] = diary_df["meseri"].map(quarter_mapping)
    diary_df["Descrizione catcon"] = diary_df["catcon"].map(condition_mapping)
    diary_df["Descrizione gsett"] = diary_df["gsett"].map(day_type_mapping)


    # Align quarter labels with the rest of this script.
    diary_df = convert_quarter_labels_to_abbreviations(diary_df)

    # Align the place label with the list used in ENG_01.
    diary_df["Luogo aggregato"] = diary_df["Luogo aggregato"].replace({
        "Luogo in esterno": "Luogo all'aperto"
    })

    diary_df["ora inizio"] = (
        diary_df["oraini"].str.zfill(2) + ":" + diary_df["minini"].str.zfill(2)
    )

    diary_df["ora fine"] = (
        diary_df["orafin"].str.zfill(2) + ":" + diary_df["minfin"].str.zfill(2)
    )

    diary_df["persona"] = (
        diary_df["profam"].astype(str) + "." + diary_df["proind"].astype(str)
    )

    # Align with the legacy pipeline:
    # the daily diary dataframe is saved after splitting rows that cross midnight.
    diary_df = split_rows_crossing_midnight(
        diary_df,
        start_column="ora inizio",
        end_column="ora fine"
    )

    save_pickle_gzip(
        diary_df,
        input_dir / "df_diario_giornaliero_full.pkl.gz"
    )

    return diary_df



# === Main function to create dataframe with 144 ts for aggregated activities and aggregated places
def build_144_timestep_dataframe(
        diary_df,
        category_column,
        categories,
        output_file_path,
        progress_description):
    """
    Build a person-by-timestep dataframe with 144 ten-minute timesteps.

    The same function is used for:
        - aggregated activities
        - aggregated occupied places

    Output format:
        persona | time_step_0 | time_step_1 | ... | time_step_143
    """
    working_df = diary_df.copy()

    category_to_code = {
        category: i + 1
        for i, category in enumerate(categories)
    }

    working_df["_start_time"] = pd.to_datetime(
        working_df["ora inizio"],
        errors="coerce"
    )

    working_df["_end_time"] = pd.to_datetime(
        working_df["ora fine"],
        errors="coerce"
    )

    # Split activities or places that cross midnight into two rows.
    crosses_midnight_mask = working_df["_start_time"] > working_df["_end_time"]

    if crosses_midnight_mask.any():
        before_midnight_df = working_df.loc[crosses_midnight_mask].copy()
        after_midnight_df = working_df.loc[crosses_midnight_mask].copy()

        before_midnight_df["_end_time"] = pd.to_datetime("23:59", format="%H:%M")
        after_midnight_df["_start_time"] = pd.to_datetime("00:00", format="%H:%M")

        working_df = pd.concat(
            [
                working_df.loc[~crosses_midnight_mask],
                before_midnight_df,
                after_midnight_df
            ],
            ignore_index=True
        )

    time_intervals = pd.date_range(
        start="00:00",
        end="23:50",
        freq="10min"
    )

    time_step_columns = [
        f"time_step_{i}"
        for i in range(len(time_intervals))
    ]

    people = working_df["persona"].drop_duplicates().tolist()

    person_timestep_dict = {
        person: [0] * len(time_intervals)
        for person in people
    }

    for person, person_df in tqdm(
            working_df.groupby("persona", sort=False),
            desc=progress_description):

        for time_step_index, time_interval in enumerate(time_intervals):
            active_rows = person_df[
                (person_df["_start_time"].dt.time <= time_interval.time()) &
                (person_df["_end_time"].dt.time > time_interval.time())
            ]

            if not active_rows.empty:
                category_value = active_rows.iloc[0][category_column]
                person_timestep_dict[person][time_step_index] = category_to_code.get(
                    category_value,
                    0
                )

    output_data = [
        [person] + time_step_values
        for person, time_step_values in person_timestep_dict.items()
    ]

    timestep_df = pd.DataFrame(
        output_data,
        columns=["persona"] + time_step_columns
    )

    timestep_df[time_step_columns] = timestep_df[time_step_columns].astype(int)

    # ---- debug check
    missing_values_count = (timestep_df[time_step_columns] == 0).sum().sum()

    if missing_values_count > 0:
        print(f"WARNING: {missing_values_count} timestep cells have value 0 because no category was mapped.")
    # ----------------

    save_pickle_gzip(
        timestep_df,
        output_file_path
    )

    return timestep_df



# Function to build all input dataframes
def rebuild_input_dataframes(input_dir):
    """
    Rebuild all input dataframes required by this script.

    Intermediate outputs:
        - df_individui_full.pkl.gz
        - df_diario_giornaliero_full.pkl.gz

    Final inputs used by ENG_01:
        - df_diario_giorn_con_individui_full.pkl.gz
        - df_144timestep_ISTAT_full.pkl.gz
        - df_luoghi_144_timestep_ISTAT_full.pkl.gz
    """
    input_dir = Path(input_dir)
    input_dir.mkdir(parents=True, exist_ok=True)

    print("-" * 60)
    print("Rebuilding ENG_01 input dataframes from ISTAT source files")
    print("-" * 60)

    print("1/5 - Building full individuals dataframe...")
    individuals_df = build_individuals_dataframe(input_dir)

    print("2/5 - Building full daily diary dataframe...")
    daily_diary_df = build_daily_diary_dataframe(input_dir)

    print("3/5 - Building 144-timestep activity dataframe...")
    activity_144_timestep_df = build_144_timestep_dataframe(
        diary_df=daily_diary_df,
        category_column="Attività aggregata",
        categories=aggregated_activities,
        output_file_path=input_dir / "df_144timestep_ISTAT_full.pkl.gz",
        progress_description="Building activity 144-timestep dataframe"
    )

    print("4/5 - Building 144-timestep occupied-place dataframe...")
    place_144_timestep_df = build_144_timestep_dataframe(
        diary_df=daily_diary_df,
        category_column="Luogo aggregato",
        categories=aggregated_places,
        output_file_path=input_dir / "df_luoghi_144_timestep_ISTAT_full.pkl.gz",
        progress_description="Building occupied-place 144-timestep dataframe"
    )

    print("5/5 - Merging daily diary and individuals dataframes...")
    diary_with_individuals_df = pd.merge(
        daily_diary_df,
        individuals_df,
        on="persona",
        how="inner"
    )

    save_pickle_gzip(
        diary_with_individuals_df,
        input_dir / "df_diario_giorn_con_individui_full.pkl.gz"
    )

    print("-" * 60)
    print("Input dataframe rebuild completed")
    print(f"Diary + individuals dataframe shape: {diary_with_individuals_df.shape}")
    print(f"Activity 144-timestep dataframe shape: {activity_144_timestep_df.shape}")
    print(f"Place 144-timestep dataframe shape: {place_144_timestep_df.shape}")
    print("-" * 60)

    return (
        diary_with_individuals_df,
        activity_144_timestep_df,
        place_144_timestep_df
    )


## FUNCTION FOR DATA LOADING
def load_data(rebuild_input_dataframes_flag=RUN_AGGR_DATASETS):
    """
    Load the input dataframes required by ENG_01.

    If one or more required input pickle files are missing, or if
    REBUILD_INPUT_DATAFRAMES is set to True, the function rebuilds them
    from the original ISTAT source files before loading them.
    """
    input_dir = Path(input_folder)

    required_input_files = {
        "diary_with_individuals": input_dir / "df_diario_giorn_con_individui_full.pkl.gz",
        "activity_144_timestep": input_dir / "df_144timestep_ISTAT_full.pkl.gz",
        "place_144_timestep": input_dir / "df_luoghi_144_timestep_ISTAT_full.pkl.gz",
    }

    missing_files = [
        file_path
        for file_path in required_input_files.values()
        if not file_path.exists()
    ]

    if rebuild_input_dataframes_flag or missing_files:
        if missing_files:
            print("One or more required input files are missing:")
            for missing_file in missing_files:
                print(f" - {missing_file}")
            print("Starting input dataframe rebuild...")
        else:
            print("REBUILD_INPUT_DATAFRAMES=True. Starting forced input dataframe rebuild...")

        rebuild_input_dataframes(input_dir)

    description_dict = pd.read_excel(
        input_dir / "descrizione_dati_TUS.xlsx"
    )

    diary_with_individuals_df = load_pickle_gzip(
        required_input_files["diary_with_individuals"]
    )

    activity_144_timestep_df = load_pickle_gzip(
        required_input_files["activity_144_timestep"]
    )

    place_144_timestep_full_df = load_pickle_gzip(
        required_input_files["place_144_timestep"]
    )

    return (
        description_dict,
        diary_with_individuals_df,
        activity_144_timestep_df,
        place_144_timestep_full_df
    )


# -------------------------------------------------------------------------------








# ----------------------------------------------------------
#       Sec 1.6 -- 1) FUNCTIONS FOR LOADING, AGGREGATION AND FILTERS
# ----------------------------------------------------------

## FUNCTION FOR APPLYING FILTERS TO THE DATAFRAME
def apply_filters(
        diary_df, 
        day_type, 
        quarter,
        regions,                                                # to filter by multiple regions
        # region,                                               # to filter by a single region
        household_types,
        household_positions):                                   # , fascia_eta): # Age group could also be added if individuals should be characterized by age (e.g., retired single vs young working single)
    filtered_df = diary_df[
        (diary_df['Descrizione gsett'] == day_type) & 
        (diary_df['Trimestre'] == quarter) &
        (diary_df['rip'].astype(str).isin(regions)) &           # use isin() to filter by multiple regions
        #(df_diario['rip'].astype(str) == str(region)) &        # to filter by a single region
        #(df_diario['classeta2'] == fascia_eta) &               # age group of the respondent can also be added to the filter (e.g., over 75)
        diary_df['tipnu2'].astype(str).str.zfill(2).isin([str(x).zfill(2) for x in household_types]) &
        diary_df['rpnuc2'].astype(str).str.zfill(2).isin([str(x).zfill(2) for x in household_positions])
    ]
    return filtered_df



## FUNCTION FOR CREATING THE AGGREGATED DATAFRAME (ACTIVITIES AND PLACES) USING VECTORIZATION
def create_aggregated_df_vectorized(df, categories):
    n_timesteps = 144
    n_cat = len(categories)
    # Build the list of columns where data for each timestep is stored.
    time_cols = [f'time_step_{t}' for t in range(n_timesteps)]
    
    
    # Extract data from the specified columns into a NumPy array.
    # Values are assumed to be integers from 1 to n_cat.
    data = df[time_cols].values.astype(int)
    
    # Create an empty matrix for counts: (n_timesteps, n_cat)
    agg = np.zeros((n_timesteps, n_cat), dtype=int)
    
    for t in range(n_timesteps):
        # Extract the values at timestep t for all individuals
        values = data[:, t]
        # Filter out any invalid values (as a safety check, e.g., if the value is < 1)
        valid_values = values[values >= 1]
        
        # Calculate counts using np.bincount.
        # Since values range from 1 to n_cat, set minlength to n_cat+1 
        # and then discard the count at position 0.
        if valid_values.size > 0:
            counts = np.bincount(valid_values, minlength=n_cat+1)[1:]
        else:
            counts = np.zeros(n_cat, dtype=int)
        agg[t, :] = counts
    
    # Create a DataFrame for aggregated counts, where each row corresponds to a timestep
    aggregated_df = pd.DataFrame(agg, columns=categories)
    return aggregated_df
# -------------------------------------------------------------------



# ----------------------------------------------------------
#       Sec 1.6 -- 2) FUNCTIONS FOR TRANSITION MATRICES AND INITIAL PROBABILITIES
# ----------------------------------------------------------
if RUN_MATRICES:

    ## FUNCTION FOR CREATING FOLDERS WHERE TRANSITION MATRICES ARE SAVED
    def create_region_folder(base_dir, region_abbr):
        """Create (if it does not exist) a subfolder for the specified region."""
        region_folder = os.path.join(base_dir, f"Mtx_{region_abbr}")
        os.makedirs(region_folder, exist_ok=True)
        return region_folder
      

    ## FUNCTION FOR CALCULATING TRANSITION MATRICES
    def calculate_transition_matrices_and_initial_probabilities(df, categories, assigned_region_name, week_day_type,quarter_abbr, num_time_steps=144):
        # num_time_steps = 144
        num_categories = len(categories)
        transition_matrix = np.zeros((num_time_steps, num_categories, num_categories))
        initial_probabilities = np.zeros((num_categories, 1))
        
        # Calculate initial probabilities
        for j in range(1, num_categories + 1):
            count_j_start = len(df[df['time_step_0'] == j])                             # it counts how many times that occupied place/activity occurs across all individuals at timestep 0 (00:00)
            initial_probabilities[j - 1, 0] = count_j_start / len(df['time_step_0'])
        
        if np.nansum(initial_probabilities) > 0:
            initial_probabilities /= np.nansum(initial_probabilities)
        else:
            initial_probabilities.fill(1 / num_categories)
        
        for t in tqdm(range(num_time_steps - 1), 
                    desc=f"Calculating transition matrices and initial probabilities (area {assigned_region_name}) for day {week_day_type}, quarter {quarter_abbr}"):        
            for i in range(num_categories):
                for j in range(num_categories):
                    count_ij = len(df[(df[f'time_step_{t}'] == i + 1) & (df[f'time_step_{t + 1}'] == j + 1)])
                    count_i = len(df[df[f'time_step_{t}'] == i + 1])
                    
                    if count_i > 0:
                        transition_matrix[t, i, j] = count_ij / count_i
                    else:
                        transition_matrix[t, i, j] = 0

                if np.nansum(transition_matrix[t, i, :]) > 0:
                    transition_matrix[t, i, :] /= np.nansum(transition_matrix[t, i, :])
                else:
                    transition_matrix[t, i, :] = 1 / num_categories

                transition_matrix[t, i, :] = np.nan_to_num(transition_matrix[t, i, :])
                assert np.isclose(np.sum(transition_matrix[t, i, :]), 1), f"Probabilities do not sum to 1 at time_step {t}, category {i+1}: {np.sum(transition_matrix[t, i, :])}"

        # To avoid the issue of switching to new day (23:50 --> 00:00) (e.g., weekday --> Saturday)
        # Copy probabilities from timestep 142 (23:40) to timestep 143 (23:50)
        transition_matrix[143] = transition_matrix[142]
        
        return transition_matrix, initial_probabilities
    
# -------------------------------------------------------------------------------------------------------




# ----------------------------------------------------------
#       Sec 1.6 -- 3) FUNCTIONS FOR SYNTHETIC PROFILES GENERATION
# ----------------------------------------------------------

## FUNCTION FOR BUILDING SYNTHETIC PROFILES OF ACTIVITIES OR OCCUPIED PLACES (starting from initial probabilities and transition matrices)
def activity_forecast(simulated_people, category, transition_matrix, initial_probabilities, categories, initial_state=None, num_time_steps=144):
    states = [str(i) for i in range(1, len(categories) + 1)]            # it defines the possible "states" (i.e., the activities or occupied places that can occur)                  
    activity_pattern = []

    for m in range (simulated_people):
        activityList = []
        
        if initial_state is None:
            # Choose the initial state based on the initial probabilities
            start_state = np.random.choice(states, replace=True, p=initial_probabilities[:, 0])
        else:
            start_state = initial_state[m]        
        
        activityList.append(str(start_state))
    
    
        for i in range(1, num_time_steps):                              # Start from 1 because state 0 has already been defined (by the initial probability)
                # Check whether activityList is not empty
                if not activityList:
                    print(f"Error: activityList is empty. Person {m+1} - Time step {i}")
                    # break            
    
                current_state_index = int(activityList[-1]) - 1         # Subtract 1 to convert to a 0-based index
                current_time_step = i % 144                             # Get the current timestep from 0 to 143
                                
                # Access the correct transition matrix for the current timestep
                transition_probabilities = transition_matrix[current_time_step, current_state_index]
                
                if not np.isclose(np.sum(transition_probabilities), 1):
                    print(f"Error: probabilities do not sum to 1 for timestep {current_time_step} and state {current_state_index}.")
                                
                # Check whether valid transition probabilities exist
                if np.sum(transition_probabilities) == 0:
                    print(f"Error: no valid transition probability for index {current_state_index}. Person {m+1} - Iteration {i}")
                    # break
                
                # Choose the next state based on transition probabilities
                next_state = np.random.choice(states, replace=True, p=transition_probabilities)
                                    
                # Update the activity list
                activityList.append(str(next_state))
        
        activity_pattern.append(activityList)
    
    return activity_pattern


## FUNCTION FOR SELECTING THE CORRECT QUARTER IN THE SIMULATED MONTHS
def get_quarter(month):
    if month in [11, 12, 1]:
        return 'Nov-Dic-Gen'
    elif month in [2, 3, 4]:
        return 'Feb-Mar-Apr'
    elif month in [5, 6, 7]:
        return 'Mag-Giu-Lug'
    elif month in [8, 9, 10]:
        return 'Ago-Set-Ott'
    


## FUNCTION FOR SELECTING (existing) TRANSITION MATRICES AND INITIAL PROBABILITIES ACCORDING TO THE DAY OF THE YEAR AND REGION
@lru_cache(maxsize=None)
def load_transition_cached(day_name, quarter, tipnu2_value, rpnuc2_value, region, path_base):
    """
    Return transition matrix and initial probabilities for the selected case.

    This loader supports both:
        - new English file names
        - legacy Italian file names generated by previous versions of the code

    Supported examples:
        transition_matrix_activity_region_Isl_domenica_Feb-Mar-Apr_householdtype_1_member_1.npy
        initial_prob_activity_region_Isl_domenica_Feb-Mar-Apr_householdtype_1_member_1.npy

        mtx_trans_attiv_reg_Isl_domenica_Feb-Mar-Apr_tipnuc_1_membro_1.npy
        prob_iniz_attiv_reg_Isl_domenica_Feb-Mar-Apr_tipnuc_1_membro_1.npy
    """
    path_base = Path(path_base)

    # Translate the Python day name to the ISTAT/source-data day label.
    day_str = source_day_map.get(day_name, "feriale")

    # Prefixes used in new and legacy file names.
    if region in region_group_names:
        new_prefix = f"group_{region}_"
        legacy_prefix = f"grup_{region}_"
    else:
        new_prefix = f"region_{region}_"
        legacy_prefix = f"reg_{region}_"

    # Candidate matrix file names, from newest to oldest naming convention.
    matrix_file_candidates = [
        # New English naming convention
        (
            f"transition_matrix_activity_{new_prefix}"
            f"{day_str}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Hybrid legacy naming convention
        (
            f"mtx_trans_attiv_{legacy_prefix}"
            f"{day_str}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Fully legacy Italian naming convention
        (
            f"mtx_trans_attiv_{legacy_prefix}"
            f"{day_str}_{quarter}_"
            f"tipnuc_{tipnu2_value}_membro_{rpnuc2_value}.npy"
        ),
    ]

    # Candidate initial-probability file names.
    probability_file_candidates = [
        # New English naming convention
        (
            f"initial_prob_activity_{new_prefix}"
            f"{day_str}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Hybrid legacy naming convention
        (
            f"prob_iniz_attiv_{legacy_prefix}"
            f"{day_str}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Fully legacy Italian naming convention
        (
            f"prob_iniz_attiv_{legacy_prefix}"
            f"{day_str}_{quarter}_"
            f"tipnuc_{tipnu2_value}_membro_{rpnuc2_value}.npy"
        ),
    ]

    def find_matrix_file(candidate_names, file_kind):
        """
        Search first in the expected folder, then in its parent folder, then
        recursively under activity_matrices_folder. Recursive search keeps
        compatibility with legacy folder structures.
        """
        search_dirs = [
            path_base,
            path_base.parent,
            Path(activity_matrices_folder),
        ]

        # Direct search in likely folders.
        for search_dir in search_dirs:
            for candidate_name in candidate_names:
                candidate_path = Path(search_dir) / candidate_name
                if candidate_path.exists():
                    return candidate_path

        # Fallback recursive search.
        for candidate_name in candidate_names:
            matches = list(Path(activity_matrices_folder).rglob(candidate_name))
            if len(matches) > 0:
                return matches[0]

        checked_names = "\n".join([f" - {name}" for name in candidate_names])

        raise FileNotFoundError(
            f"{file_kind} not found for day={day_str}, quarter={quarter}, "
            f"householdtype={tipnu2_value}, member={rpnuc2_value}, region={region}.\n"
            f"Base folder checked: {path_base}\n"
            f"Candidate names:\n{checked_names}"
        )

    matrix_path = find_matrix_file(
        matrix_file_candidates,
        file_kind="Transition matrix"
    )

    probability_path = find_matrix_file(
        probability_file_candidates,
        file_kind="Initial probabilities"
    )

    transition_matrix = np.load(matrix_path)
    initial_activity_probabilities = np.load(probability_path)

    return transition_matrix, initial_activity_probabilities



## FUNCTION FOR CONVERTING THE FRAME INTO TIME OF DAY
def convert_to_time(frame):
    hours = frame // 6
    minutes = (frame % 6) * 10
    return f'{hours:02d}:{minutes:02d}'


## FUNCTIONS FOR CREATING AND ASSIGNING DESTINATION FOLDERS FOR TRANSITION MATRICES / INITIAL PROBABILITIES
def create_regional_folder(base_dir, region_abbr):
    region_folder = os.path.join(base_dir, f"{region_abbr}")
    os.makedirs(region_folder, exist_ok=True)
    return region_folder

# Function for creating the folder for file saving
def create_configuration_folder(type_folder, region_abbr, tipnu2_value, rpnuc2_value):
    folder_name = f"householdtype_{tipnu2_value}_member_{rpnuc2_value}"
    configuration_folder = os.path.join(type_folder, folder_name)
    os.makedirs(configuration_folder, exist_ok=True)
    return configuration_folder


## FUNCTION FOR MATCHING REGION NAMES TO THEIR GROUPING
def region_group_abbreviation(region_list):
    # Per ogni codice (es. '1') recupera il nome completo da reg_italia_dict e l'abbreviazione da region_abbr_dict
    abbreviation_list = [region_abbr_dict[italy_regions_dict[reg]] for reg in region_list]
    return "_".join(abbreviation_list)
# -------------------------------------------------------------------------------------------------------




# ----------------------------------------------------------
#       Sec 1.6 -- 4) FUNCTIONS FOR PLOTTING ACTIVITY PROFILES DISTRIBUTION
# ----------------------------------------------------------
if RUN_PLOTS:

    ## FUNCTION FOR CREATING THE ACTIVITY AND OCCUPIED-PLACE PLOT
    def create_plot(
            aggregated_df,
            categories,
            category_english_dict,
            color_map,
            title,
            file_name,
            num_ticks=12,
            num_axis_time_step=144,
            show_title=True,
            fig_size=(7.2, 4.8),
            font_size=11,
            label_font_size=12,
            tick_font_size=10,
            legend_font_size=8,
            title_font_size=12,
            alpha=0.90,
            legend_position="bottom",
            legend_ncol=4,
            save_pdf=True):
        """
        Key options:
        - show_title=False removes the title from the image, useful for papers.
        - num_ticks=12 gives one tick every 2 hours for 144 ten-minute timesteps.
        - save_pdf=True saves both PNG and PDF.
        """

        with plt.rc_context({
            "font.size": font_size,
            "axes.labelsize": label_font_size,
            "xtick.labelsize": tick_font_size,
            "ytick.labelsize": tick_font_size,
            "legend.fontsize": legend_font_size,
            "axes.titlesize": title_font_size,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none"
        }):

            fig, ax = plt.subplots(figsize=fig_size)

            n = len(aggregated_df)
            x = np.arange(n)
            bottom = np.zeros(n, dtype=float)

            for category_name in categories:
                if category_name in aggregated_df.columns:
                    values = aggregated_df[category_name].values.astype(float)
                    color = color_map.get(category_name, "#999999")
                    label_engl = category_english_dict.get(category_name, category_name)

                    ax.fill_between(
                        x,
                        bottom,
                        bottom + values,
                        label=label_engl,
                        color=color,
                        alpha=alpha,
                        linewidth=0.15,
                        edgecolor="white"
                    )

                    bottom = bottom + values

            # Generate labels for each timestep
            if pd.api.types.is_datetime64_any_dtype(aggregated_df.index) or isinstance(aggregated_df.index, pd.DatetimeIndex):
                timesteps = [idx.strftime("%H:%M") for idx in aggregated_df.index]
            else:
                try:
                    timesteps = [convert_to_time(int(i)) for i in aggregated_df.index]
                except Exception:
                    timesteps = [
                        f"{i // 6:02d}:{(i % 6) * 10:02d}"
                        for i in range(num_axis_time_step)
                    ]

            # X-axis ticks
            step = max(1, n // num_ticks)
            tick_positions = list(range(0, n, step))
            tick_labels = [timesteps[i] for i in tick_positions]

            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, ha="center")

            ax.set_xlim(0, n - 1)
            ax.set_ylim(0, max(bottom) * 1.02)

            ax.set_xlabel("Hour of the day")
            ax.set_ylabel("Number of simulated people")

            if show_title:
                ax.set_title(title, pad=10)

            ax.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.45)
            ax.grid(axis="x", visible=False)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            handles, labels = ax.get_legend_handles_labels()

            if legend_position == "right":
                ax.legend(
                    handles,
                    labels,
                    loc="center left",
                    bbox_to_anchor=(1.02, 0.5),
                    frameon=False,
                    handlelength=1.2,
                    borderaxespad=0.0
                )

            elif legend_position == "bottom":
                ax.legend(
                    handles,
                    labels,
                    loc="upper center",
                    bbox_to_anchor=(0.5, -0.18),
                    ncol=legend_ncol,
                    frameon=False,
                    handlelength=1.2,
                    columnspacing=1.0,
                    labelspacing=0.6
                )

            output_path = Path(output_pictures_folder) / file_name

            fig.savefig(
                output_path,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04
            )

            if save_pdf:
                pdf_output_path = output_path.with_suffix(".pdf")
                fig.savefig(
                    pdf_output_path,
                    bbox_inches="tight",
                    pad_inches=0.04
                )

            plt.close(fig)
    
    #--------------------------------------------------------------------------------------


## 1.6.2 F(X) Debug and validation

In [ ]:
"""
    This section inlcudes some functions suited for sanity check between new aggregated 
    dataframes and the ones provided by the authors (derived from Italian 2013 ISTAT Time-Use-Surveys).

    If new TUS data are available (e.g., Eurostat-based, or more recent), it is useful to check they
    have same structure as the original ones.

    NOTE: Be aware that if you decide to modify the aggregation structure between activities (or locations), 
    you have to modify and adjust also the input file "descrizione_dati_TUS.xlsx" (i.e., description of TUS data),
    where you can define the aggregation scheme and the information related to data is provided.

"""


# ----------------------------------------------------------
#       Sec 1.6.2 -- 1)     Optional validation -- Compare rebuilt inputs against legacy inputs
# ----------------------------------------------------------

# Function to compare old and new dataframe derived from aggregating TUS data
def compare_dataframes(old_df, new_df, dataframe_name, max_differences_to_show=20):
    """
    Compare two dataframes and print a readable report.

    This is intended as a regression test:
    it verifies whether the new unified input-building pipeline reproduces
    the same outputs previously generated by the legacy scripts.
    """
    print("-" * 80)
    print(f"Comparing dataframe: {dataframe_name}")
    print("-" * 80)

    print(f"Old shape: {old_df.shape}")
    print(f"New shape: {new_df.shape}")

    if old_df.shape != new_df.shape:
        print("RESULT: DIFFERENT SHAPE")
        return False

    if list(old_df.columns) != list(new_df.columns):
        print("RESULT: DIFFERENT COLUMNS")

        old_columns = list(old_df.columns)
        new_columns = list(new_df.columns)

        only_old = [col for col in old_columns if col not in new_columns]
        only_new = [col for col in new_columns if col not in old_columns]

        print(f"Columns only in old dataframe: {only_old}")
        print(f"Columns only in new dataframe: {only_new}")

        return False

    try:
        pd.testing.assert_frame_equal(
            old_df,
            new_df,
            check_dtype=True,
            check_exact=True
        )

        print("RESULT: IDENTICAL")
        return True

    except AssertionError as error:
        print("RESULT: DIFFERENT CONTENT")
        print("")
        print("Pandas comparison message:")
        print(error)
        print("")

        # Try to identify and print a small sample of differences.
        try:
            comparison_mask = old_df.ne(new_df)

            different_cells = comparison_mask.stack()
            different_cells = different_cells[different_cells]

            print(f"Number of different cells: {len(different_cells)}")

            if len(different_cells) > 0:
                print("")
                print(f"First {max_differences_to_show} different cells:")

                for row_index, column_name in different_cells.index[:max_differences_to_show]:
                    old_value = old_df.loc[row_index, column_name]
                    new_value = new_df.loc[row_index, column_name]

                    print(
                        f"Row index: {row_index} | Column: {column_name} | "
                        f"Old: {old_value} | New: {new_value}"
                    )

        except Exception as diff_error:
            print("Could not print cell-level differences.")
            print(f"Reason: {diff_error}")

        return False

# ------------------------------------------------------------------------------------------





# ----------------------------------------------------------
#       Sec 1.6.2 -- 2)     Optional validation -- INDEPENDENT SANITY CHECK
# ----------------------------------------------------------

# Function to compare and evaluate sanity check of input dataframes
def validate_rebuilt_input_dataframes(
        diary_with_individuals_df,
        activity_144_timestep_df,
        place_144_timestep_df):
    """
    Run independent sanity checks on the rebuilt ENG_01 input dataframes.

    These checks do not compare against the legacy outputs.
    They verify whether the rebuilt dataframes are structurally consistent
    with the assumptions used later in ENG_01.
    """
    print("=" * 80)
    print("Independent validation of rebuilt input dataframes")
    print("=" * 80)

    time_step_columns = [
        f"time_step_{i}"
        for i in range(144)
    ]

    # ----------------------
    # 1) Required columns
    # ----------------------
    print("")
    print("1) Checking required columns...")

    required_diary_columns = [
        "persona",
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2"
    ]

    missing_diary_columns = [
        col for col in required_diary_columns
        if col not in diary_with_individuals_df.columns
    ]

    missing_activity_columns = [
        col for col in ["persona"] + time_step_columns
        if col not in activity_144_timestep_df.columns
    ]

    missing_place_columns = [
        col for col in ["persona"] + time_step_columns
        if col not in place_144_timestep_df.columns
    ]

    print(f"Missing diary columns: {missing_diary_columns}")
    print(f"Missing activity timestep columns: {missing_activity_columns[:10]}")
    print(f"Missing place timestep columns: {missing_place_columns[:10]}")

    # ----------------------
    # 2) Quarter and day labels
    # ----------------------
    print("")
    print("2) Checking day-type and quarter labels...")

    expected_day_types = set(source_day_types)
    expected_quarters = set(quarter_abbr_dict.values())

    actual_day_types = set(
        diary_with_individuals_df["Descrizione gsett"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    actual_quarters = set(
        diary_with_individuals_df["Trimestre"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    print(f"Expected day types: {sorted(expected_day_types)}")
    print(f"Actual day types:   {sorted(actual_day_types)}")
    print(f"Unexpected day types: {sorted(actual_day_types - expected_day_types)}")
    print(f"Missing day types:    {sorted(expected_day_types - actual_day_types)}")

    print(f"Expected quarters: {sorted(expected_quarters)}")
    print(f"Actual quarters:   {sorted(actual_quarters)}")
    print(f"Unexpected quarters: {sorted(actual_quarters - expected_quarters)}")
    print(f"Missing quarters:    {sorted(expected_quarters - actual_quarters)}")

    # ----------------------
    # 3) Timestep value ranges
    # ----------------------
    print("")
    print("3) Checking timestep value ranges...")

    activity_values = activity_144_timestep_df[time_step_columns]
    place_values = place_144_timestep_df[time_step_columns]

    activity_min = activity_values.min().min()
    activity_max = activity_values.max().max()
    place_min = place_values.min().min()
    place_max = place_values.max().max()

    print(f"Activity timestep min value: {activity_min}")
    print(f"Activity timestep max value: {activity_max}")
    print(f"Expected activity range: 1 to {len(aggregated_activities)}")

    print(f"Place timestep min value: {place_min}")
    print(f"Place timestep max value: {place_max}")
    print(f"Expected place range: 1 to {len(aggregated_places)}")

    activity_zero_count = int((activity_values == 0).sum().sum())
    place_zero_count = int((place_values == 0).sum().sum())

    print(f"Activity cells equal to 0: {activity_zero_count}")
    print(f"Place cells equal to 0: {place_zero_count}")

    if activity_zero_count > 0:
        print("WARNING: Some activity timestep cells are equal to 0.")
        print("This usually means that at least one activity label was not mapped correctly.")

    if place_zero_count > 0:
        print("WARNING: Some place timestep cells are equal to 0.")
        print("This usually means that at least one place label was not mapped correctly.")

    # ----------------------
    # 4) Person coverage
    # ----------------------
    print("")
    print("4) Checking person coverage across dataframes...")

    diary_people = set(diary_with_individuals_df["persona"].dropna().astype(str).unique())
    activity_people = set(activity_144_timestep_df["persona"].dropna().astype(str).unique())
    place_people = set(place_144_timestep_df["persona"].dropna().astype(str).unique())

    print(f"People in diary + individuals dataframe: {len(diary_people)}")
    print(f"People in activity timestep dataframe:  {len(activity_people)}")
    print(f"People in place timestep dataframe:     {len(place_people)}")

    print(f"People in activity but not in diary: {len(activity_people - diary_people)}")
    print(f"People in place but not in diary:    {len(place_people - diary_people)}")
    print(f"People in diary but not in activity: {len(diary_people - activity_people)}")
    print(f"People in diary but not in place:    {len(diary_people - place_people)}")

    # ----------------------
    # 5) NaN checks on mapped columns
    # ----------------------
    print("")
    print("5) Checking missing values in key mapped columns...")

    key_mapped_columns = [
        "Attività aggregata",
        "Luogo aggregato",
        "Trimestre",
        "Descrizione gsett"
    ]

    for column in key_mapped_columns:
        if column in diary_with_individuals_df.columns:
            missing_count = int(diary_with_individuals_df[column].isna().sum())
            print(f"{column}: {missing_count} missing values")
        else:
            print(f"{column}: column not found")

    print("")
    print("Independent validation completed.")
    print("=" * 80)



# Function to compare df_diario_giornaliero_full between old and new configuration
def compare_dataframes_ignoring_row_order(old_df, new_df, dataframe_name):
    """
    Compare two dataframes as unordered collections of rows.

    This is useful when the same rows may appear in a different order.
    """
    print("=" * 80)
    print(f"Comparing dataframe ignoring row order: {dataframe_name}")
    print("=" * 80)

    if old_df.shape != new_df.shape:
        print(f"Different shape: old={old_df.shape}, new={new_df.shape}")
        return False

    if list(old_df.columns) != list(new_df.columns):
        print("Different columns or different column order.")
        return False

    old_normalized = old_df.copy()
    new_normalized = new_df.copy()

    # Convert all columns to string-compatible values for robust row hashing.
    # NaN values are replaced by a stable placeholder.
    old_normalized = old_normalized.fillna("__NA__").astype(str)
    new_normalized = new_normalized.fillna("__NA__").astype(str)

    old_hashes = pd.util.hash_pandas_object(
        old_normalized,
        index=False
    ).value_counts().sort_index()

    new_hashes = pd.util.hash_pandas_object(
        new_normalized,
        index=False
    ).value_counts().sort_index()

    try:
        pd.testing.assert_series_equal(
            old_hashes,
            new_hashes,
            check_names=False
        )

        print("RESULT: SAME ROWS, POSSIBLY DIFFERENT ORDER")
        return True

    except AssertionError as error:
        print("RESULT: DIFFERENT ROW SET")
        print("")
        print(error)
        return False
    

# Function to compare people sets within "df_diario_giornaliero_full"
def compare_people_sets_by_filter_group(legacy_df, rebuilt_df):
    group_cols = [
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2",
    ]

    legacy_groups = (
        legacy_df
        .groupby(group_cols)["persona"]
        .apply(lambda x: frozenset(x.dropna().astype(str).unique()))
        .reset_index(name="legacy_people")
    )

    rebuilt_groups = (
        rebuilt_df
        .groupby(group_cols)["persona"]
        .apply(lambda x: frozenset(x.dropna().astype(str).unique()))
        .reset_index(name="rebuilt_people")
    )

    comparison = pd.merge(
        legacy_groups,
        rebuilt_groups,
        on=group_cols,
        how="outer"
    )

    comparison["same_people"] = (
        comparison["legacy_people"] == comparison["rebuilt_people"]
    )

    different_groups = comparison[~comparison["same_people"]]

    print(f"Number of groups with different people sets: {len(different_groups)}")

    if len(different_groups) > 0:
        print(different_groups[group_cols].head(30))

    return different_groups


## 1.6.3 F(x) weekly activity profile generation

In [21]:
# Function for collect information from activity profile dataset
def parse_activity_profile_filename(file_name):
    """
    Extract region, household type, household member and suffix from both
    English and legacy Italian annual-profile file names.

    Supported examples:
        synthetic_activity_profiles_2023_200_pp_region_Isl_householdtype_1_member_1_coppiaCF_caponucleo.xlsx

        profili_sintetici_attività_2023_200_pp_reg_Isl_tipnuc_1_membro_1_coppiaCF_caponucleo.xlsx
    """
    name_body = Path(file_name).stem
    parts = name_body.split("_")

    def value_after(possible_labels):
        for label in possible_labels:
            if label in parts:
                idx = parts.index(label)
                if idx + 1 < len(parts):
                    return parts[idx + 1], idx
        return None, None

    region_value, region_label_idx = value_after(["region", "reg"])
    household_type_value, household_type_label_idx = value_after(["householdtype", "tipnuc"])
    member_value, member_label_idx = value_after(["member", "membro"])

    if region_value is None or household_type_value is None or member_value is None:
        raise ValueError(
            f"Could not parse region/household/member information from file name: {file_name}"
        )

    # The suffix starts after the household-member value.
    suffix_start_idx = member_label_idx + 2
    suffix = "_".join(parts[suffix_start_idx:]) if suffix_start_idx < len(parts) else "unknown"

    return {
        "name_body": name_body,
        "region": region_value,
        "tipnu2_value": int(household_type_value),
        "rpnuc2_value": int(member_value),
        "suffix": suffix,
    }



# Function for builduing a correct name between Italian or English dataframes input files
def build_legacy_weekly_output_name(profile_info):
    """
    Build a legacy-compatible weekly output file name.

    Example:
        pr_At_St_2023_200_pp_reg_Isl_tipnuc_1_membro_1_coppiaCF_caponucleo.xlsx
    """
    name_body = profile_info["name_body"]

    if name_body.startswith("profili_sintetici_attività_"):
        out_body = name_body.replace("profili_sintetici_attività_", "pr_At_St_")

    elif name_body.startswith("synthetic_activity_profiles_"):
        out_body = name_body.replace("synthetic_activity_profiles_", "pr_At_St_")
        out_body = out_body.replace("_region_", "_reg_")
        out_body = out_body.replace("_householdtype_", "_tipnuc_")
        out_body = out_body.replace("_member_", "_membro_")

    else:
        # Fallback: keep the original name body and add a weekly prefix.
        out_body = f"pr_At_St_{name_body}"

    return f"{out_body}.xlsx"


def read_annual_profile_excel(input_path):
    """
    Read an annual synthetic profile Excel file.

    Supports both:
        - English sheet name: Profiles
        - Italian sheet name: Profili

    If neither is found, the first sheet is used.
    """
    excel_file = pd.ExcelFile(input_path)

    if "Profiles" in excel_file.sheet_names:
        profile_sheet = "Profiles"
    elif "Profili" in excel_file.sheet_names:
        profile_sheet = "Profili"
    else:
        profile_sheet = excel_file.sheet_names[0]

    df = pd.read_excel(
        input_path,
        sheet_name=profile_sheet,
        parse_dates=["Datetime"]
    )

    if "Datetime" not in df.columns:
        raise ValueError(
            f"'Datetime' column not found in {input_path}, sheet '{profile_sheet}'"
        )

    df = df.set_index("Datetime")

    return df

 ## 1.7 Load input data (and possible sanity check in aggregated input data)

Loaded input aggregated data. Possibility to compare newly generated dataframes with original ones provided by the authors (to check the structure is the same)

In [ ]:
# ----------------------------------------------------------
#       Sec 1.7 -- 1.0) AGGREGATED INPUT DATAFRAME CREATION 
# ----------------------------------------------------------
# Section to rebuild aggregated dataframes and load them (if RUN_AGGR_DATASETS = False, only loading existing dfs)
description_dict, diary_with_individuals_df, activity_144_timestep_df, place_144_timestep_full_df = load_data()

# ----------------------------------------------------------







# ----------------------------------------------------------
#       Sec 1.7 -- 1.1) OPTIONAL DEBUGGING:
#       validation of rebuilt aggregated TUS input files
# ----------------------------------------------------------
#
# Enable with:
#     RUN_DEBUG_INPUTS = True
#
# Goal:
#     Compare the rebuilt input dataframes with the legacy ones and check whether
#     they are suitable for the ENG_01 workflow.
#
# Key interpretation:
#     - Strict dataframe equality is useful but too severe: it fails if row order
#       or non-operative labels differ.
#     - For df_diario_giorn_con_individui_full.pkl.gz, the most important check is
#       whether apply_filters() would select the same people for each combination of:
#           Descrizione gsett, Trimestre, rip, tipnu2, rpnuc2.
#     - Small differences in df_144timestep_ISTAT_full.pkl.gz may be acceptable if
#       they replace legacy 0 values with valid activity codes, e.g. 7 = Spostamenti.
# ----------------------------------------------------------

if RUN_DEBUG_INPUTS:

    print("=" * 80)
    print("OPTIONAL DEBUGGING OF REBUILT INPUT DATAFRAMES")
    print("=" * 80)

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.0) Load legacy and rebuilt diary-with-individuals dataframes
    # ------------------------------------------------------

    legacy_diary_df = load_pickle_gzip(
        Path("input") / "01_input_profiles_att_OLD" / "df_diario_giorn_con_individui_full.pkl.gz"
    )

    rebuilt_diary_df = load_pickle_gzip(
        Path(input_folder) / "df_diario_giorn_con_individui_full.pkl.gz"
    )

    print("")
    print("Loaded diary-with-individuals dataframes:")
    print(f"Legacy shape:  {legacy_diary_df.shape}")
    print(f"Rebuilt shape: {rebuilt_diary_df.shape}")

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.1) Optional row-set comparison
    # ------------------------------------------------------
    #
    # Checks whether the two diary dataframes contain exactly the same rows,
    # ignoring row order. This can still fail if some labels/codes were normalized
    # differently in the rebuilt pipeline. It can also be memory-intensive.
    # ------------------------------------------------------

    # compare_dataframes_ignoring_row_order(
    #     old_df=legacy_diary_df,
    #     new_df=rebuilt_diary_df,
    #     dataframe_name="df_diario_giorn_con_individui_full.pkl.gz"
    # )

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.2) Row-by-row diagnostic comparison of operational columns
    # ------------------------------------------------------
    #
    # Diagnostic only. If "persona" differs in most rows, the two dataframes are
    # probably sorted differently, so high difference percentages are expected.
    # ------------------------------------------------------

    operational_columns = [
        "persona",
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2",
        "Attività aggregata",
        "Luogo aggregato",
        "ora inizio",
        "ora fine",
    ]

    print("")
    print("1) Row-by-row diagnostic comparison of operational columns")
    print("-" * 80)

    for col in operational_columns:
        if col not in legacy_diary_df.columns:
            print(f"{col}: missing in legacy dataframe")
            continue

        if col not in rebuilt_diary_df.columns:
            print(f"{col}: missing in rebuilt dataframe")
            continue

        same_values = (
            legacy_diary_df[col].fillna("__NA__").astype(str)
            .eq(rebuilt_diary_df[col].fillna("__NA__").astype(str))
        )

        different_count = int((~same_values).sum())
        different_share = different_count / len(same_values) * 100

        print(f"{col}: {different_count} different rows ({different_share:.4f}%)")

    print("Note: large percentages here often indicate different row order.")

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.3) Filter-group person-count comparison
    # ------------------------------------------------------
    #
    # Checks whether each model-relevant filter group contains the same number of
    # people in legacy and rebuilt data.
    # ------------------------------------------------------

    group_cols = [
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2",
    ]

    legacy_counts = (
        legacy_diary_df
        .groupby(group_cols)["persona"]
        .nunique()
        .reset_index(name="legacy_people")
    )

    rebuilt_counts = (
        rebuilt_diary_df
        .groupby(group_cols)["persona"]
        .nunique()
        .reset_index(name="rebuilt_people")
    )

    counts_comparison = pd.merge(
        legacy_counts,
        rebuilt_counts,
        on=group_cols,
        how="outer"
    ).fillna(0)

    counts_comparison["difference"] = (
        counts_comparison["rebuilt_people"] - counts_comparison["legacy_people"]
    )

    different_count_groups = counts_comparison[
        counts_comparison["difference"] != 0
    ]

    print("")
    print("2) Filter-group person-count comparison")
    print("-" * 80)
    print(f"Number of groups with different person counts: {len(different_count_groups)}")

    if len(different_count_groups) > 0:
        print(different_count_groups.head(30))
    else:
        print("OK: all filter groups contain the same number of people.")

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.4) Filter-group person-set comparison
    # ------------------------------------------------------
    #
    # Stronger than the previous check: verifies that each filter group contains
    # exactly the same people, not only the same number of people.
    # ------------------------------------------------------

    print("")
    print("3) Filter-group person-set comparison")
    print("-" * 80)

    different_people_groups = compare_people_sets_by_filter_group(
        legacy_diary_df,
        rebuilt_diary_df
    )

    if len(different_people_groups) == 0:
        print("OK: all filter groups contain exactly the same people.")
    else:
        print("WARNING: some filter groups contain different people.")
        print(different_people_groups.head(30))


    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.6) Independent structural validation
    # ------------------------------------------------------
    #
    # Does not rely on the legacy files. Checks required columns, valid labels,
    # timestep ranges, people coverage, and missing mapped values.
    # ------------------------------------------------------

    print("")
    print("5) Independent structural validation")
    print("-" * 80)

    validate_rebuilt_input_dataframes(
        diary_with_individuals_df=diary_with_individuals_df,
        activity_144_timestep_df=activity_144_timestep_df,
        place_144_timestep_df=place_144_timestep_full_df
    )

    # ------------------------------------------------------
    #   Sec 1.7 -- 1.1.7) Compact summary
    # ------------------------------------------------------

    print("")
    print("6) Compact summary")
    print("-" * 80)

    activity_values = activity_144_timestep_df.iloc[:, 1:]
    place_values = place_144_timestep_full_df.iloc[:, 1:]

    activity_zero_count = int((activity_values == 0).sum().sum())
    place_zero_count = int((place_values == 0).sum().sum())

    print(f"Activity cells equal to 0: {activity_zero_count}")
    print(f"Place cells equal to 0:    {place_zero_count}")

    print(
        f"Activity timestep min/max: "
        f"{activity_values.min().min()} / {activity_values.max().max()} "
        f"(expected 1 / {len(aggregated_activities)})"
    )

    print(
        f"Place timestep min/max:    "
        f"{place_values.min().min()} / {place_values.max().max()} "
        f"(expected 1 / {len(aggregated_places)})"
    )

    print("")
    print("Quarter labels:")
    print(diary_with_individuals_df["Trimestre"].unique())

    print("")
    print("Day-type labels:")
    print(diary_with_individuals_df["Descrizione gsett"].unique())

    print("")
    print("OPTIONAL DEBUGGING COMPLETED")
    print("=" * 80)

# ------------------------------------------------------------

## 1.8 Build transition matrices, initial probabilities, and plots

 Define matrix, probability, and plot calculations for the selected individual classes, days, quarters, and regions.

In [23]:
# Create synthetic profiles for each source day type (weekday, Saturday, Sunday)
daily_activity_df_list = []
case_list = []



# If Markov transition matrices should not be calculated and activity plots should not be printed, skip this entire section of code
if RUN_MATRICES == False and RUN_PLOTS == False:
    print("Transition matrices are not calculated and daily activity plots are not printed")


# At least one of the two outputs is required (matrices or activity plots)
else:
    # ----------------------------------------------------------
    #       Sec 1.8 -- 1.1) CASE FOR AGGREGATED REGION GROUPS (e.g., South+Islands): 
    #       WARNING: uncomment where "single REGION" to have the output for each separated geographical area (e.g., South AND Islands)
    #                and comment similar code lines used for region groups
    # ----------------------------------------------------------

    for group_name, region_list in region_groups.items():
    # for region_idx in italy_regions_dict.keys():                                                  # single REGION
        group_abbr = region_group_abbreviation(region_list)
        # # Retrieve the full region name and abbreviation
        # region_name = italy_regions_dict[region_idx]      # E.g., 'North-West'                    # single REGION
        # region_abbr = region_abbr_dict[region_name]        # E.g., 'N-O'                          # single REGION
        
        group_timestamp = datetime.datetime.now()
        # region_timestamp = datetime.datetime.now()                                                # single REGION
        print('-' * 20)
        print(f"Time: {group_timestamp} - Calculating matrices and probabilities for {group_name} ({group_abbr})")
        
        # Create the regional structure: folders dedicated to the region group
        group_activity_folder = create_regional_folder(activity_matrices_folder, group_abbr)
        group_place_folder = create_regional_folder(location_matrices_folder, group_abbr)
        # # regional_activity_folder = create_regional_folder(activity_matrices_folder, region_abbr)       # single REGION
        # # regional_place_folder = create_regional_folder(location_matrices_folder, region_abbr)            # single REGION


        # Iterate over household-type and household-member configurations
        for household_type, household_position in individual_configurations:
            print('-' * 40)
            print(f"Creating transition matrices and transition probabilities for household_type={household_type}, household_position={household_position}")
            print('-' * 20)
            
            # Set the current configurations
            household_type_list = [household_type]
            household_position_list = [household_position]
        
            # Retrieve filter values
            rpnuc2_value = household_position_list[0]
            tipnu2_value = household_type_list[0]
            
            # # Retrieve the descriptions corresponding to the filter values
            tipnu2_description = description_dict.loc[description_dict['tipnu2'] == tipnu2_value, 'tipo di nucleo'].iloc[0]
            rpnuc2_description = description_dict.loc[description_dict['rpnuc2'] == rpnuc2_value, 'posizione nel nucleo familiare'].iloc[0]
        
            # Create folders dedicated to the activity and place configurations
            # activity_configuration_folder = create_configuration_folder(regional_activity_folder, region_abbr, tipnu2_value, rpnuc2_value)        # single REGION
            # place_configuration_folder = create_configuration_folder(regional_place_folder, region_abbr, tipnu2_value, rpnuc2_value)              # single REGION
            activity_configuration_folder = create_configuration_folder(group_activity_folder, group_abbr, tipnu2_value, rpnuc2_value)
            place_configuration_folder = create_configuration_folder(group_place_folder, group_abbr, tipnu2_value, rpnuc2_value)
            
            # Iterate over quarter and day (using product for clarity)
            for quarter, day_type in product(source_quarters, source_day_types):
                
                    quarter_abbr = quarter_abbr_dict.get(quarter, 'Unknown')    
                    week_day_type = day_type
                    
                    # Apply filters to the dataframe
                    filtered_diary_with_individuals_df = apply_filters(
                        diary_with_individuals_df, 
                        week_day_type, 
                        quarter_abbr, 
                        # region_idx,                                                                   # single REGION
                        region_list,
                        household_type_list, 
                        household_position_list
                        )
                    
                    # Count the number of filtered individuals
                    filtered_individuals = filtered_diary_with_individuals_df['persona'].unique()
                    num_individuals = len(filtered_individuals)
                    print('-'*20)
                    print(f"For area {group_abbr} (config: household_type {tipnu2_value}, household_position {rpnuc2_value}), {num_individuals} individuals were used.")
                
                
                    # If there are no individuals, record the case with flag 1 and skip the iteration
                    if num_individuals == 0:
                        case_list.append([
                            group_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                            quarter_abbr,
                            week_day_type,
                            num_individuals,
                            1  # flag: no data
                        ])
                        print(f"--- No data found for {group_abbr}, household_type {tipnu2_value}, household_position {rpnuc2_value}, {quarter_abbr} and {week_day_type}. Iteration skipped. ---")
                        # print(f"--- No data found for {region_name}, household_type {tipnu2_value}, household_position {rpnuc2_value}, {quarter_abbr} and {week_day_type}. Iteration skipped. ---") # single REGION
                        continue
                    else:
                        # If individuals are available, record the case with flag 0
                        case_list.append([
                            group_abbr,
                            # region_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                            quarter_abbr,
                            week_day_type,
                            num_individuals,
                            0  # flag: data available
                        ])

                    # Filter the 144-timestep dataframes
                    filtered_activity_144_timestep_df = activity_144_timestep_df[activity_144_timestep_df['persona'].isin(filtered_individuals)]
                    filtered_place_144_timestep_df = place_144_timestep_full_df[place_144_timestep_full_df['persona'].isin(filtered_individuals)]
                    # Create aggregated dataframes
                    aggregated_activity_df = create_aggregated_df_vectorized(filtered_activity_144_timestep_df, aggregated_activities)
                    aggregated_place_df = create_aggregated_df_vectorized(filtered_place_144_timestep_df, aggregated_places)
                    #------------------------------------------------------------------------------------------------------------------------------                    



                    # ----------------------------------------------------------
                    #       Sec 1.8 -- 1.2) PRINT ACTIVITY AND OCCUPIED-PLACE PLOTS
                    # ----------------------------------------------------------
                    if RUN_PLOTS:
                        if not os.path.exists(output_folder):
                            os.makedirs(output_folder)
                        
                        # Get the current time and format it without seconds
                        now = datetime.datetime.now()
                        current_date_str = now.strftime('%Y-%m-%d_%H-%M')
                        print("Creating aggregated activity and place plots...")
                        
                        
                        activity_plot_title = (
                            f'Aggregated daily activities - {day_english_dict[week_day_type]} - {quarter_abbr_english_dict[quarter_abbr]} '
                            f'| Area: {group_abbr} | Family type: {suffix_english_dict[tipnu2_value, rpnuc2_value]}'
                        )

                        activity_base_plot_name = (
                            f'activities_{day_english_dict[week_day_type]}_{quarter_abbr_english_dict[quarter_abbr]}_'
                            f'householdtype_{tipnu2_value}_member_{household_position}_ENG'
                        )

                        place_plot_title = (
                            f'Aggregated occupied places - {day_english_dict[week_day_type]} - {quarter_abbr_english_dict[quarter_abbr]} '
                            f'| Area: {group_abbr} | Family type: {suffix_english_dict[tipnu2_value, rpnuc2_value]}'
                        )

                        place_base_plot_name = (
                            f'places_{day_english_dict[week_day_type]}_{quarter_abbr_english_dict[quarter_abbr]}_'
                            f'householdtype_{tipnu2_value}_member_{rpnuc2_value}_ENG'
                        )

                        # # ------------------------------------------------------------------
                        # # ACTIVITY PLOT - single-column version (suited for 85 mm)
                        # # ------------------------------------------------------------------
                        # create_plot(
                        #     aggregated_activity_df,
                        #     aggregated_activities,
                        #     activity_english_dict,
                        #     activity_color_map,
                        #     activity_plot_title,
                        #     f'{activity_base_plot_name}_single_column.png',
                        #     show_title=False,
                        #     num_ticks=8,
                        #     fig_size=(3.55, 3.10),
                        #     font_size=8,
                        #     label_font_size=8.5,
                        #     tick_font_size=7.5,
                        #     legend_font_size=5.8,
                        #     legend_position="bottom",
                        #     legend_ncol=3,
                        #     save_pdf=True
                        # )

                        # ------------------------------------------------------------------
                        # ACTIVITY PLOT - two-column / full-width version (suited for 170-180 mm)
                        # ------------------------------------------------------------------
                        create_plot(
                            aggregated_activity_df,
                            aggregated_activities,
                            activity_english_dict,
                            activity_color_map,
                            activity_plot_title,
                            f'{activity_base_plot_name}_two_column.png',
                            show_title=False,
                            num_ticks=12,
                            fig_size=(7.20, 3.70),
                            font_size=9.5,
                            label_font_size=10.5,
                            tick_font_size=9,
                            legend_font_size=7.2,
                            legend_position="right",
                            legend_ncol=1,
                            save_pdf=True
                        )


                        # # ------------------------------------------------------------------
                        # # OCCUPIED-PLACE PLOT - single-column version (UNCOMMENT if to be printed)
                        # # ------------------------------------------------------------------
                        # create_plot(
                        #     aggregated_place_df,
                        #     aggregated_places,
                        #     place_english_dict,
                        #     place_color_map,
                        #     place_plot_title,
                        #     f'{place_base_plot_name}_single_column.png',
                        #     show_title=False,
                        #     num_ticks=8,
                        #     fig_size=(3.55, 3.10),
                        #     font_size=8,
                        #     label_font_size=8.5,
                        #     tick_font_size=7.5,
                        #     legend_font_size=5.8,
                        #     legend_position="bottom",
                        #     legend_ncol=3,
                        #     save_pdf=True
                        # )

                        # # ------------------------------------------------------------------
                        # # OCCUPIED-PLACE PLOT - two-column / full-width version (UNCOMMENT if to be printed)
                        # # ------------------------------------------------------------------
                        # create_plot(
                        #     aggregated_place_df,
                        #     aggregated_places,
                        #     place_english_dict,
                        #     place_color_map,
                        #     place_plot_title,
                        #     f'{place_base_plot_name}_two_column.png',
                        #     show_title=False,
                        #     num_ticks=12,
                        #     fig_size=(7.20, 3.70),
                        #     font_size=9.5,
                        #     label_font_size=10.5,
                        #     tick_font_size=9,
                        #     legend_font_size=7.2,
                        #     legend_position="right",
                        #     legend_ncol=1,
                        #     save_pdf=True
                        # )
                        

                    #------------------------------------------------------------------------------------------------------------------------------



                    # ----------------------------------------------------------
                    #       Sec 1.8 -- 1.3) CREATE TRANSITION MATRICES AND INITIAL PROBABILITIES
                    # ----------------------------------------------------------
                    if RUN_MATRICES:
                        
                        # Build file names including the regional abbreviation
                        activity_transition_matrix_file_name = f"transition_matrix_activity_group_{group_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
                        activity_initial_prob_file_name = f"initial_prob_activity_group_{group_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
                        place_transition_matrix_file_name = f"transition_matrix_places_group_{group_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
                        place_initial_prob_file_name = f"initial_prob_places_group_{group_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
                                        
                        print('-'*20)
                        print(f"Iteration over group {group_abbr}: saving in {os.path.join(activity_configuration_folder, activity_transition_matrix_file_name)}")
                        # print(f"Iteration over group {group_abbr}: saving in {os.path.join(place_configuration_folder, place_transition_matrix_file_name)}")  
                        
                        # Calculate transition matrices and initial probabilities for places and activities
                        activity_transition_matrix, initial_activity_probabilities = calculate_transition_matrices_and_initial_probabilities(filtered_activity_144_timestep_df, aggregated_activities, group_abbr,week_day_type,quarter_abbr)
                        place_transition_matrix, initial_place_probabilities = calculate_transition_matrices_and_initial_probabilities(filtered_place_144_timestep_df, aggregated_places, group_abbr,week_day_type,quarter_abbr)
                        
                        # Save the files inside the folder dedicated to the region
                        np.save(os.path.join(activity_configuration_folder, activity_transition_matrix_file_name), activity_transition_matrix)
                        np.save(os.path.join(place_configuration_folder, place_transition_matrix_file_name), place_transition_matrix)
                        np.save(os.path.join(activity_configuration_folder, activity_initial_prob_file_name), initial_activity_probabilities)
                        np.save(os.path.join(place_configuration_folder, place_initial_prob_file_name), initial_place_probabilities)
                        

                        # ----------------------------------------------------------
                        #       Sec 1.8 -- 1.4) CASE FOR SINGLE REGION (uncomment all the following block with "single REGION")
                        # ----------------------------------------------------------
                        # # activity_transition_matrix_file_name = f"mtx_trans_activity_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"        
                        # place_transition_matrix_file_name = f"mtx_trans_places_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"        
                        # # activity_initial_prob_file_name = f"initial_prob_activity_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"        
                        # place_initial_prob_file_name = f"initial_prob_places_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"        
                        # print(f"Iteration over area {region_idx}: saving in {os.path.join(place_configuration_folder, place_transition_matrix_file_name)}")
                        # print(f"Iteration over area {region_idx}: saving in {os.path.join(activity_configuration_folder, activity_transition_matrix_file_name)}")
                        # activity_transition_matrix, initial_activity_probabilities = calculate_transition_matrices_and_initial_probabilities(filtered_activity_144_timestep_df, aggregated_activities, region_name)        
                        # place_transition_matrix, initial_place_probabilities = calculate_transition_matrices_and_initial_probabilities(filtered_place_144_timestep_df, aggregated_places, region_name)        
                        
                        # # Save transition matrices
                        # # np.save(os.path.join(activity_configuration_folder, f'mtx_trans_activity_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy'), activity_transition_matrix)        
                        # np.save(os.path.join(place_configuration_folder, f'mtx_trans_places_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy'), place_transition_matrix)        
                        
                        # # Save initial probabilities
                        # # np.save(os.path.join(activity_configuration_folder, f'initial_prob_activity_reg_{region_abbr}_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy'), initial_activity_probabilities)        
                        # np.save(os.path.join(place_configuration_folder, f'initial_prob_places_{week_day_type}_{quarter_abbr}_householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy'), initial_place_probabilities)        
                    
                    #------------------------------------------------------------------------------------------------------------------------------      


--------------------
Time: 2026-07-02 16:23:39.623975 - Calculating matrices and probabilities for Group1 (N-O_N-E_Cen)
----------------------------------------
Creating transition matrices and transition probabilities for household_type=0, household_position=0
--------------------
--------------------
For area N-O_N-E_Cen (config: household_type 0, household_position 0), 419 individuals were used.
Creating aggregated activity and place plots...
--------------------
For area N-O_N-E_Cen (config: household_type 0, household_position 0), 392 individuals were used.
Creating aggregated activity and place plots...
--------------------
For area N-O_N-E_Cen (config: household_type 0, household_position 0), 421 individuals were used.
Creating aggregated activity and place plots...
--------------------
For area N-O_N-E_Cen (config: household_type 0, household_position 0), 403 individuals were used.
Creating aggregated activity and place plots...
--------------------
For area N-O_N-E_Cen (config

 ## 1.9 Create synthetic (annual) activity profiles from Markov chains

 Used transition matrices and initial probabilities to build annual synthetic profiles for a defined number of individuals and for the selected individual classes.

In [24]:
"""         
    ---    DESCRIPTION  ---
    How the code works:  
        1. Define how many people synthetic profiles should be created for (simulated_people)  
        2. For each person, the activity/occupied place at timestep 0 (e.g., 00:00) is defined by randomly sampling from the initial-probability "matrix"  
        3. For each timestep, the code iterates and determines the state transition (e.g., transition from activity "j" to activity "m" using the probabilities from the previously calculated transition matrix).  
        4. Creation of a set of "if/elif" statements in which all possible combinations are defined  
        e.g., if [1,2,3] = [Sleeping, Eating, Work]  
            - if starting from 1 (Sleeping), probability of 12 (Sleeping --> Eating) or 13 (Sleeping --> Work)  
            - if starting from 2 (Eating), probability of 21 (Eating --> Sleeping) or 23 (Eating --> Work)  
            - if starting from 3 (Work), probability of 31 (Work --> Sleeping) or 32 (Work --> Eating)
"""


# Initialize a dictionary to count occurrences of each initial state
initial_state_counts = {state: 0 for state in range(1, len(aggregated_activities) + 1)}

# Determine the number of activities (dataset columns) and the daily timesteps (144 x 10 minutes)
num_activities = len(aggregated_activities)

# Build a profile for the year with 10-minute resolution
# datetime_index = pd.date_range(start=f'1/1/{simulation_year}', selected_periods=num_time_steps*simulation_days, freq='10T')
datetime_index = pd.date_range(start=f"{simulation_year}-01-01 00:00:00", periods=num_time_steps * simulation_days, freq="10min")

# Determine the day of the week for January 1
start_day_name = datetime_index[0].day_name()


# ----------------------------------------------------------
#       Sec 1.9 -- 1) CREATE ANNUAL SYNTHETIC PROFILES FOR INDIVIDUAL CATEGORIES 
#-----------------------------------------------------------
if RUN_YEARLY_PROFILES:

    # Check
    print(f'January 1 {simulation_year} is a {start_day_name}.')
    print('-'*20)
    print("Start creating synthetic profiles...")
    print('-'*20)

    # Iterate over household-type and household-member configurations
    for household_type, household_position in individual_configurations:
        # Set the current configurations
        household_type_list = [household_type]
        household_position_list = [household_position]
        
        # Retrieve filter values
        rpnuc2_value = household_position_list[0]
        tipnu2_value = household_type_list[0]
        
        # # Retrieve the descriptions corresponding to the filter values
        tipnu2_description = description_dict.loc[description_dict['tipnu2'] == tipnu2_value, 'tipo di nucleo'].iloc[0]
        rpnuc2_description = description_dict.loc[description_dict['rpnuc2'] == rpnuc2_value, 'posizione nel nucleo familiare'].iloc[0]
        
        markov_chain_timestamp = datetime.datetime.now()
        print('-' * 20)
        print(f"Start creating synthetic activity profiles for the selected individuals (category: household_type = {tipnu2_value}, member = {rpnuc2_value}) using Markov chains: {markov_chain_timestamp}")


        # ----------------------------------------------------------
        #       Sec 1.9 -- 1.1)  SELECT MATRICES AND PROBABILITIES BY GEOGRAPHICAL AREA
        #-----------------------------------------------------------
        for region_code, region_name in italy_regions_dict.items():
            region_abbr = region_abbr_dict[region_name]
            
            # 1) Build the path to the matrices according to the "aggregated" logic
            if (tipnu2_value, rpnuc2_value) in [(3,1), (3,3)]:
                # categories always aggregated into two macro-groups
                if region_code in region_groups['Group1']:
                    input_matrices_folder = activity_matrices_folder / 'N-O_N-E_Cen' / f'householdtype_{tipnu2_value}_member_{rpnuc2_value}'
                    region_for_file = 'N-O_N-E_Cen'
                else:
                    input_matrices_folder = activity_matrices_folder / 'Sud_Isl' / f'householdtype_{tipnu2_value}_member_{rpnuc2_value}'
                    region_for_file = 'Sud_Isl'

            elif (tipnu2_value, rpnuc2_value) in [(4,1), (4,3)]:
                # categories aggregated only for South and Islands
                if region_code in region_groups['Group2']:
                    input_matrices_folder = activity_matrices_folder / 'Sud_Isl' / f'householdtype_{tipnu2_value}_member_{rpnuc2_value}'
                    region_for_file = 'Sud_Isl'
                else:
                    input_matrices_folder = activity_matrices_folder / region_abbr / f'householdtype_{tipnu2_value}_member_{rpnuc2_value}' #/ region_abbr
                    region_for_file = region_abbr

            else:
                # all other categories have an area-specific folder
                input_matrices_folder = activity_matrices_folder / region_abbr / f'householdtype_{tipnu2_value}_member_{rpnuc2_value}' #/ region_abbr
                region_for_file = region_abbr

            print(f"\n-- Category ({tipnu2_value},{rpnuc2_value}) – Area {region_name}: folder = {input_matrices_folder}")
                        
                    
            # Initialize a matrix to count occurrences of each activity at each timestep
            activity_counts_per_timestep = np.zeros((num_time_steps*simulation_days, len(aggregated_activities)), dtype=int)
            

            # ----------------------------------------------------------
            #       Sec 1.9 -- 1.2)  NUMPY PRE-ALLOCATION SECTION for matrices and related objects used to build synthetic profiles
            #-----------------------------------------------------------

            # 1) NumPy pre-allocation
            total_steps = simulation_days * num_time_steps
            profiles = np.empty((simulated_people, total_steps), dtype=np.int16)
            activity_counts = np.zeros((total_steps, len(aggregated_activities)), dtype=int)
            states = np.arange(1, len(aggregated_activities) + 1)
            
            # 2) Simulation for each person
            for person in range(simulated_people):
                prev_state = None
                write_idx = 0
            
                for day in range(simulation_days):
                    # Data, day, quarter
                    date = datetime_index[day * num_time_steps]
                    day_name = date.day_name()
                    quarter = get_quarter(date.month)
            
                    # Load from cache
                    key = (day_name, quarter, tipnu2_value, rpnuc2_value, region_for_file)
                    mtx, prob_init = load_transition_cached(*key, input_matrices_folder)
            
                    # Initial state of the day
                    if prev_state is None:
                        start_state = np.random.choice(states, p=prob_init[:, 0])
                    else:
                        start_state = prev_state
            
                    # First timestep
                    profiles[person, write_idx] = start_state
                    current_state = start_state
            
                    # Remaining timesteps
                    for t in range(1, num_time_steps):
                        # mtx shape (144, N_states, N_states)
                        trans_p = mtx[(t - 1) % num_time_steps, current_state - 1]
                        next_state = np.random.choice(states, p=trans_p)
                        profiles[person, write_idx + t] = next_state
                        current_state = next_state
            
                    # save for the next day
                    prev_state = current_state
                    write_idx += num_time_steps
            
            # 3) (Optional) aggregated counts
            for t in range(total_steps):
                vals, cnts = np.unique(profiles[:, t], return_counts=True)
                activity_counts[t, vals - 1] = cnts
            
            # 4) Convert from NumPy to DataFrame and save
            df_profiles = pd.DataFrame(
                profiles.T,
                index=datetime_index,
                columns=[f'person_{i+1}' for i in range(simulated_people)]
            )
            #-------------------------------------------------------------------------


            
            # Create the column names "person_1", "person_2", ..., "person_n"
            column_names = [f'person_{i+1}' for i in range(simulated_people)]
            
            # Rename the DataFrame columns with the new names
            df_profiles.columns = column_names
            
            
            # Create a DataFrame from activity counts at each timestep
            activity_counts_df = pd.DataFrame(activity_counts_per_timestep, columns=aggregated_activities)
            
            # Save the DataFrame to a CSV file
            now = datetime.datetime.now()
            current_date_str = now.strftime('%Y-%m-%d_%H-%M')
            
            
            # Full path for the Excel file
            profile_saving_folder = Path(yearly_activity_profiles_folder) / region_abbr
            os.makedirs(profile_saving_folder, exist_ok=True)
            
            suffix = suffix_dict[(tipnu2_value, rpnuc2_value)]

            file_name = (
            f'synthetic_activity_profiles_'
            f'{simulation_year}_'
            f'{simulated_people}_pp_'
            f'region_{region_abbr}_'
            f'householdtype_{tipnu2_value}_'
            f'member_{rpnuc2_value}'
            f'_{suffix}.xlsx'
            )
            excel_path = profile_saving_folder / file_name
            
            
            # Create an Excel writer using openpyxl as the engine
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                # Write the profile DataFrame to the first worksheet
                df_profiles.to_excel(writer, sheet_name='Profiles', index_label='Datetime')
                
                # Create a DataFrame with additional information
                info_data = {
                    'Description': [
                        'Simulated year',
                        'Number of people for whom a synthetic profile was generated',
                        'Family category', 
                        'Family member',
                        'Region considered'
                        ],
                    
                    'Value': [
                        simulation_year,
                        simulated_people,
                        tipnu2_description, 
                        rpnuc2_description,
                        region_abbr],
                    
                    'Code': [
                        '',  # Leave blank for "Simulated year"
                        '',  # Leave blank for "Number of people"
                        tipnu2_value, 
                        rpnuc2_value,
                        '']  # Leave blank for "Region considered"]
                }
                df_info = pd.DataFrame(info_data)
                
                # Add a more detailed description in a separate cell
                description = f"The created profiles belong to family category '{tipnu2_description}', for family member '{rpnuc2_description}', for region {region_abbr}."
            
                # Write the information DataFrame to the second worksheet
                df_info.to_excel(writer, sheet_name='Information', index=False)
                
                # Add the detailed description to the second worksheet
                writer.sheets['Information'].cell(row=len(df_info)+3, column=1).value = description

 # **01B. Extract weekly activity profiles for selected periods**

 This section extracts activity profiles for a selected number of individuals for the weeks of interest defined at the beginning of the code (one per season).  
 WARNING: if the periods are changed, electric-load profiles must also be adapted (e.g., Space Cooling).

 ## 1.10 Create weekly activity profiles

In [25]:
# ----------------------------------------------------------
#       Sec 1.10 -- CREATE WEEKLY ACTIVITY PROFILES
# ----------------------------------------------------------
#
# This section extracts selected weeks from existing annual synthetic profiles.
#
# Compatibility:
#   - Reads new English annual-profile names:
#       synthetic_activity_profiles_*.xlsx
#
#   - Reads legacy Italian annual-profile names:
#       profili_sintetici_attività_*.xlsx
#
#   - Does NOT read pr_At_St_*.xlsx files, because those are already weekly
#     profile outputs generated by the legacy Italian code.
#
# Output:
#   By default, weekly files are saved with the legacy-compatible prefix:
#       pr_At_St_...  (profiles_activity_week)
#
# This keeps compatibility with downstream code that may still expect the old
# Italian file naming convention.
# ----------------------------------------------------------


if RUN_WEEKLY_PROFILES:

    for region_name, region_abbr in region_abbr_dict.items():
        input_dir = os.path.join(yearly_activity_profiles_folder, region_abbr)

        if not os.path.isdir(input_dir):
            print(f"⚠️ Folder not found for {region_name}, skipping...")
            continue

        # Annual-profile input patterns.
        # Do not include pr_At_St_*.xlsx here: those are already weekly outputs.
        patterns = [
            os.path.join(input_dir, "synthetic_activity_profiles_*.xlsx"),
            os.path.join(input_dir, "profili_sintetici_attività_*.xlsx"),
        ]

        files = []

        for pattern in patterns:
            files.extend(glob.glob(pattern))

        if not files:
            print(f"⚠️ No annual profile file found in {input_dir}")
            continue

        output_dir_region = os.path.join(weekly_activity_profiles_folder, region_abbr)
        os.makedirs(output_dir_region, exist_ok=True)

        for input_path in tqdm(files, desc=f"Region {region_abbr}"):
            print("=" * 40)

            try:
                fname = os.path.basename(input_path)

                profile_info = parse_activity_profile_filename(fname)

                tipnu2_value = profile_info["tipnu2_value"]
                rpnuc2_value = profile_info["rpnuc2_value"]
                suffix = profile_info["suffix"]

                # Load annual profile dataframe.
                df = read_annual_profile_excel(input_path)

                out_file_name = build_legacy_weekly_output_name(profile_info)
                out_path = os.path.join(output_dir_region, out_file_name)

                if len(out_path) > 240:
                    print(f"⚠️ Path too long ({len(out_path)}), file skipped.")
                    continue

                with pd.ExcelWriter(
                        out_path,
                        engine="openpyxl",
                        datetime_format="yyyy-mm-dd HH:MM:SS") as writer:

                    # 1) Extract selected periods.
                    for sheet, (start_s, end_s) in selected_periods.items():
                        df_slice = df.loc[start_s:end_s]

                        if df_slice.empty:
                            print(f"   – No data for '{sheet}' in {fname}")
                        else:
                            df_slice.to_excel(writer, sheet_name=sheet[:31])

                    # 2) Information sheet.
                    type_description = suffix_dict.get(
                        (tipnu2_value, rpnuc2_value),
                        suffix
                    )

                    info_data = {
                        "Description": [
                            "Simulated year",
                            "Simulated people",
                            "Family category",
                            "Family member",
                            "Region considered",
                            "Source annual profile file"
                        ],
                        "Value": [
                            simulation_year,
                            simulated_people,
                            type_description,
                            type_description.split("_")[-1],
                            region_abbr,
                            fname
                        ],
                        "Code": [
                            "",
                            "",
                            tipnu2_value,
                            rpnuc2_value,
                            "",
                            ""
                        ]
                    }

                    df_info = pd.DataFrame(info_data)
                    df_info.to_excel(writer, sheet_name="Information", index=False)

                    ws_info = writer.sheets["Information"]
                    description_text = (
                        f"The profiles belong to category '{type_description}', "
                        f"member '{type_description.split('_')[-1]}', "
                        f"region {region_abbr}."
                    )

                    ws_info.cell(row=len(df_info) + 3, column=1).value = description_text

                print(f"✅ Saved: {out_path}")
                print("-" * 20)

            except Exception as e:
                print(
                    f"❌ Processing error for region {region_abbr} "
                    f"with respect to input file {input_path}.\n\n"
                    f"Error message: {e}"
                )
                print("-" * 20)
                continue

else:
    print("Weekly activity profiles are not extracted.")


Weekly activity profiles are not extracted.


## 1.XXX Conclusion

In [26]:
#---------------------------------------------------------------------------------------------------------------------------------------------

# Get the end time
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")

End time : 2026-07-02 16:25:20.066531
Execution time : 0:02:09.342126
